# EmoSense Classifier Training, feature selection and visualization for each participant
 1. Baseline Normalization Pipeline
 2. Trains EN / SVM / RF classifiers per emotion (each emotion vs. baseline).
 3. feature importances are computed and stored for downstream analysis.
 4. Compute Correlations among features including rank correlation and clustering 
 5. Visualize feature clusters
 6. Extract emotion clusters based on ranking 
 7. Analyze consensus networks 


## 1. Imports

In [ ]:
import os
import pickle
import logging
import warnings
import re
from copy import deepcopy
from datetime import datetime
from collections import Counter
from IPython.display import display


import numpy as np
import pandas as pd
import scipy.io as sio
import networkx as nx
import scipy.stats as stats
from scipy.stats import zscore, iqr, sem

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D

import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from scipy.cluster.hierarchy import fcluster, linkage
from sklearn.metrics import silhouette_score

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, roc_auc_score, accuracy_score
)

from joblib import Parallel, delayed
import shap

from utils.ml_utils import get_strategy

warnings.filterwarnings('ignore')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

## 2. Configuration — edit here

In [ ]:
# ── Participant ID ──────────────────────────────────────────────────────────────
PID = 'emo06'

# ── Data paths ──────────────────────────────────────────────────────────────────
DATA_ROOT = r"/home/mbijanzadeh/codes/emosense-preprocessing/data"

# -------------------Emo 5-----------------
if PID == 'emo05':
    TASK_MAT_PATH = os.path.join(
        DATA_ROOT, PID,
        "agg_data_segment_multitap_all_16-May-2025.mat"
    )
#-------------------Emo 6-----------------
elif PID == 'emo06':
    TASK_MAT_PATH = os.path.join(
        DATA_ROOT, PID,
        "agg_data_segment_multitap_all_18-May-2025.mat"
    )

# #-------------------Emo 7-----------------
elif PID == 'emo07':
    TASK_MAT_PATH = os.path.join(
        DATA_ROOT, PID,
        "agg_data_segment_multitap_all_18-May-2025.mat"
    )

# #-------------------Emo 1-----------------
elif PID == 'emo01':
    TASK_MAT_PATH = os.path.join(
        DATA_ROOT, PID,
        "agg_data_segment_multitap_all_18-May-2025.mat"
    )

#-------------------Emo 5-----------------BASE line ---------------------------------------
if PID == 'emo05':
    BASE_MAT_PATH = os.path.join(
        DATA_ROOT, PID,
        "agg_data_segment_baseline_multitap_all_23-Jun-2025.mat"
    )

#-------------------Emo 6-----------------
elif PID == 'emo06':
    BASE_MAT_PATH = os.path.join(
        DATA_ROOT, PID,
        "agg_data_segment_baseline_multitap_all_01-Feb-2026.mat"
    )

# #-------------------Emo 7-----------------
elif PID == 'emo07':
    
    BASE_MAT_PATH = os.path.join(
        DATA_ROOT, PID,
        "agg_data_segment_baseline_multitap_all_30-Apr-2026.mat"
    )

# #-------------------Emo 1-----------------
elif PID == 'emo01':
    BASE_MAT_PATH = os.path.join(
        DATA_ROOT, PID,
        "agg_data_segment_baseline_multitap_all_01-Jun-2026.mat"
    )

SAVE_PATH = os.path.join(DATA_ROOT, PID) + '/'
FIG_PATH  = os.path.join(DATA_ROOT, PID, 'paper_figures') + '/'

os.makedirs(FIG_PATH, exist_ok=True)

# ── Emotions / targets ─────────────────────────────────────────────────────────
N_EMOTIONS   = 15                       # how many emotions to include (0..14)
EMOTION_COLS = list(range(N_EMOTIONS))  # column indices into target array

# ── Training ───────────────────────────────────────────────────────────────────
N_FOLDS = 4
RANDOM_SEED = 77

# ── Visualisation ──────────────────────────────────────────────────────────────
CUSTOM_PALETTE = ['#7B1FA2', '#FB8C00', '#FDD835']   # EN / SVM / RF

timestamp = datetime.now().strftime("%Y%m%d")


print("Configuration loaded for", PID)

## 3. Data loading & feature DataFrame construction

In [ ]:
# ── Column name utilities ───────────────────────────────────────────────────────

PHYSIO_COLS_WITH_EDA = [
    'mean_hr', 'std_hr',
    'mean_eda_raw', 'std_eda_raw', 'mean_tonic', 'std_tonic', 'mean_phasic', 'std_phasic',
    'mean_sys', 'max_sys', 'min_sys', 'std_sys', 'kurt_sys',
    'mean_dia', 'max_dia', 'min_dia', 'std_dia', 'kurt_dia',
    'resp_rate', 'WL_resp', 'aac_resp', 'MFL_resp', 'mav_resp', 'rms_resp',
    'WL_egg', 'aac_egg', 'MFL_egg', 'mav_egg', 'rms_egg',
    'blink_rate',
    'mean_corrug', 'max_corrug', 'min_corrug', 'std_corrug', 'kurt_corrug',
    'vardiff_corrug', 'mav_corrug', 'rms_corrug',
    'mean_zyg', 'max_zyg', 'min_zyg', 'std_zyg', 'kurt_zyg',
    'vardiff_zyg', 'mav_zyg', 'rms_zyg',
    'trev_int_velo', 'trev_co',
    'session_num', 'trial_num', 'block_num'
]

PHYSIO_COLS_NO_EDA = [
    'mean_hr', 'std_hr',
    'mean_sys', 'max_sys', 'min_sys', 'std_sys', 'kurt_sys',
    'mean_dia', 'max_dia', 'min_dia', 'std_dia', 'kurt_dia',
    'resp_rate', 'WL_resp', 'aac_resp', 'MFL_resp', 'mav_resp', 'rms_resp',
    'WL_egg', 'aac_egg', 'MFL_egg', 'mav_egg', 'rms_egg',
    'blink_rate',
    'mean_corrug', 'max_corrug', 'min_corrug', 'std_corrug', 'kurt_corrug',
    'vardiff_corrug', 'mav_corrug', 'rms_corrug',
    'mean_zyg', 'max_zyg', 'min_zyg', 'std_zyg', 'kurt_zyg',
    'vardiff_zyg', 'mav_zyg', 'rms_zyg',
    'trev_int_velo', 'trev_co',
    'session_num', 'trial_num', 'block_num'
]


def _clean_label_string(raw_list):
    """Strip brackets and quotes from MATLAB string arrays."""
    chars_to_remove = "[']"
    return ["".join(c for c in s if c not in chars_to_remove)
            for s in [str(x) for x in raw_list]]


def build_feature_df(mat, struct_key, pid):
    """
    Build a single feature DataFrame from a MATLAB .mat struct.
    Works for both the task ('agg_data') and baseline ('base_data') structs.
    Handles the emo01 special case (no EDA sensor).

    Returns
    -------
    df : pd.DataFrame  — all features + session_num / trial_num / block_num
    """
    s = mat[struct_key]

    # --- frequency features ---
    freq_feat = pd.DataFrame(s['freq_feat'][0, 0])
    freq_feat.columns = _clean_label_string(s['freq_feat_label'][0, 0].flatten())

    # --- physiological features ---
    
    
# 1. Define your master list of all features
    


     # --- physiological features ---
    if pid == 'emo01':   # no EDA
        arrays  = ['HRV', 'bp', 'resp', 'egg', 'eog', 'emg', 'trev',
                   'session_num', 'trial_num', 'block_num']
        columns = PHYSIO_COLS_NO_EDA
    else:
        arrays  = ['HRV', 'eda', 'bp', 'resp', 'egg', 'eog', 'emg', 'trev',
                   'session_num', 'trial_num', 'block_num']
        columns = PHYSIO_COLS_WITH_EDA

    parts = []
    for key in arrays:
        data = s[key][0, 0]
        if key in ('session_num', 'trial_num', 'block_num'):
            data = data.astype(int)
        parts.append(data)


    physio_feat = pd.DataFrame(np.hstack(parts), columns=columns)

    print(physio_feat.columns)

    df = pd.concat([freq_feat, physio_feat], axis=1)
    return df


def drop_nan_rows(df, ref_indices=None):
    """
    Drop rows with any NaN.  If ref_indices is supplied, also return the
    NaN index positions so they can be used to align a target array.
    """
    nan_idx = df[df.isnull().any(axis=1)].index
    dropped = df.drop(index=nan_idx).reset_index(drop=True)
    return (dropped, nan_idx) if ref_indices is not None else dropped

## 4. Block-level z-score normalisation (task + baseline together)

In [ ]:
all_feat_norm.head()

## 5. Target extraction

## 6. Cross-validation fold builder

In [ ]:
def subsample_balanced(X_scaled, y, seed=None):
    """Subsample X_scaled and y to balance classes without replacement."""
    # Convert to DataFrame for easier indexing
    X_df = pd.DataFrame(X_scaled, index=y.index if hasattr(y, 'index') else range(len(y)))
    
    # Calculate class distribution and minimum size
    counts = Counter(y)
    min_size = min(counts.values())
    
    # Set random seed for reproducibility
    if seed is not None:
        np.random.seed(seed)
    
    # Subsample each class
    X_balanced = pd.DataFrame()
    y_balanced = []
    for label in counts.keys():
        class_indices = X_df.index[y == label]
        sampled_indices = np.random.choice(class_indices, size=min_size, replace=False)
        X_balanced = pd.concat([X_balanced, X_df.loc[sampled_indices]])
        y_balanced.extend([label] * min_size)
    
    # Convert back to NumPy arrays
    X_balanced = X_balanced.to_numpy()
    y_balanced = np.array(y_balanced)
    
    return X_balanced, y_balanced

## fold_train

In [ ]:
from collections import Counter


def get_folds_data_fixed_train(
    n_split=4,
    feat_pd=None,
    target=None,
    n_iterations=5,
    emotion_name="",          # used in skip/warning messages
):
    """
    Block-stratified cross-validation that guarantees **both classes appear
    in every fold's train AND test set**.

    Strategy (by construction, no post-hoc patching)
    -------------------------------------------------
    1.  Find all blocks that contain at least one class-1 sample  (M blocks).
    2.  Feasibility check:
          • Need M >= 2  (one class-1 block in test, ≥1 remaining in train).
          • Need n_split <= M  (each fold's test must receive its own class-1
            block; blocks are never shared between test sets).
        If M < 2  → impossible for any n_split; skip this emotion.
        If n_split > M → reduce n_split to M (fewest folds that still work).
    3.  Shuffle the M class-1 blocks and assign exactly ONE to each fold's
        test set.  This guarantees test has class 1 and train has ≥ M-1 ≥ 1
        class-1 blocks.
    4.  Distribute the remaining blocks (class-0-only + any surplus class-1)
        round-robin across folds to balance fold sizes.
    5.  Scale (fit on train only), then generate n_iterations balanced
        subsamples per fold via subsample_balanced().
    """

    target = np.asarray(target)
    train_df = feat_pd.copy()

    unique_blocks = np.array(sorted(np.unique(train_df["block_num"].tolist())))
    total_blocks = len(unique_blocks)
    print(f"Total number of blocks: {total_blocks}")

    # ── Per-block class inventory ─────────────────────────────────────────
    block_class_map = {
        blk: set(np.unique(target[train_df["block_num"] == blk]))
        for blk in unique_blocks
    }
    print("Class distribution per block:", {k: sorted(v) for k, v in block_class_map.items()})

    blocks_with_1 = [b for b, c in block_class_map.items() if 1 in c]
    M = len(blocks_with_1)

    # ── Feasibility check ────────────────────────────────────────────────
    if M < 2:
        # Even 1 fold can't have class 1 in both train and test
        print(
            f"[SKIP] Emotion '{emotion_name}': only {M} block(s) contain class 1 — "
            "cannot guarantee both classes in train AND test for any fold."
        )
        return None, None, None, None

    if n_split > M:
        print(
            f"Reducing n_split from {n_split} to {M} "
            f"(only {M} blocks contain class 1)."
        )
        n_split = M

    # ── Assign blocks to folds BY CONSTRUCTION ───────────────────────────
    # Shuffle class-1 blocks; assign one to each fold's test set
    class1_blocks = list(blocks_with_1)
    np.random.shuffle(class1_blocks)

    # All other blocks (class-0-only + surplus class-1 blocks beyond n_split)
    reserved_class1 = class1_blocks[:n_split]          # one per fold
    surplus_class1  = class1_blocks[n_split:]           # extra class-1 blocks
    other_blocks    = [b for b in unique_blocks if b not in reserved_class1]
    remaining       = surplus_class1 + [b for b in other_blocks if b not in surplus_class1]
    np.random.shuffle(remaining)

    # Start each fold's test set with its reserved class-1 block
    fold_test_blocks = [[reserved_class1[i]] for i in range(n_split)]

    # Distribute remaining blocks round-robin for balanced fold sizes
    for idx, blk in enumerate(remaining):
        fold_test_blocks[idx % n_split].append(blk)

    # ── Sanity check (should always pass by construction) ────────────────
    for fi in range(n_split):
        test_b  = fold_test_blocks[fi]
        train_b = [b for b in unique_blocks if b not in test_b]
        test_cls  = set().union(*(block_class_map[b] for b in test_b))
        train_cls = set().union(*(block_class_map[b] for b in train_b))
        if not ({0, 1} <= test_cls) or not ({0, 1} <= train_cls):
            # Should never reach here; raise so the caller knows
            raise RuntimeError(
                f"Fold {fi+1} class-balance guarantee violated. "
                f"test classes={test_cls}, train classes={train_cls}. "
                f"Emotion: '{emotion_name}'"
            )

    # ── Build datasets ───────────────────────────────────────────────────
    scaler = MinMaxScaler(feature_range=(0, 1))
    x_train, y_train, x_test, y_test = [], [], [], []

    for fold_idx in range(n_split):
        test_blocks  = fold_test_blocks[fold_idx]
        train_blocks = [b for b in unique_blocks if b not in test_blocks]

        print(f"\nFold {fold_idx + 1}: Test blocks = {len(test_blocks)}, Train blocks = {len(train_blocks)}")

        train_mask = train_df["block_num"].isin(train_blocks)
        test_mask  = train_df["block_num"].isin(test_blocks)

        X_train_ = train_df.loc[train_mask]
        X_test_  = train_df.loc[test_mask]
        Y_train_ = target[X_train_.index]
        Y_test_  = target[X_test_.index]

        print(f"X_train_ rows: {X_train_.shape[0]}, Y_train_ rows: {len(Y_train_)}")
        print(f"X_test_  rows: {X_test_.shape[0]},  Y_test_  rows: {len(Y_test_)}")
        assert X_train_.shape[0] == len(Y_train_), "Train shape mismatch"
        assert X_test_.shape[0]  == len(Y_test_),  "Test shape mismatch"

        # Scale: fit on train only
        X_train_scaled = scaler.fit_transform(X_train_.iloc[:, :-3])
        X_test_scaled  = scaler.transform(X_test_.iloc[:, :-3])

        for iteration in range(n_iterations):
            X_tr_bal, Y_tr_bal = subsample_balanced(
                X_train_scaled, Y_train_,
                seed=fold_idx * n_iterations + iteration,
            )
            X_te_bal, Y_te_bal = subsample_balanced(
                X_test_scaled, Y_test_,
                seed=fold_idx * n_iterations + iteration + 1000,
            )
            x_train.append(X_tr_bal); y_train.append(Y_tr_bal)
            x_test.append(X_te_bal);  y_test.append(Y_te_bal)

            print(f"Fold {fold_idx + 1} iteration {iteration}:")
            print(f"   Blocks in test:  {test_blocks}")
            print(f"   Blocks in train: {train_blocks}")
            print(f"   y_train before balancing: {Counter(Y_train_)}")
            print(f"   y_train after  balancing: {Counter(Y_tr_bal)}")
            print(f"   y_test  before balancing: {Counter(Y_test_)}")
            print(f"   y_test  after  balancing: {Counter(Y_te_bal)}")

    print(f"\nTotal datasets generated: {len(x_train)} (expected {n_split * n_iterations})")
    assert len(x_train) == n_split * n_iterations, "Output length mismatch"

    return x_train, y_train, x_test, y_test

## 7. Model definitions & hyperparameter grids

In [ ]:
MODEL_NAMES = ['EN', 'SVM', 'RF']

MODELS = {
    'EN':  LogisticRegression(penalty='elasticnet', solver='saga', max_iter = 5000),
    'SVM': SVC(kernel='linear', probability=True, random_state=RANDOM_SEED),
    'RF':  RandomForestClassifier(random_state=RANDOM_SEED, oob_score=True, criterion='entropy'),
}

PARAM_GRIDS = {
    'EN':  {'l1_ratio': np.linspace(0, 1, 5), 'C': [0.5, 1, 5, 10]},
    'SVM': {'C': [0.5, 1, 5, 10]},
    'RF':  {'n_estimators': [100, 300], 'max_depth': [1, 3, 6, None]},
}

## 8. Core ML utilities

In [ ]:
# ── Grid search ─────────────────────────────────────────────────────────────────

def _n_best(cv_results, n=1):
    """Return parameters, mean score and std for the nth-best CV result."""
    idx    = np.where(cv_results['rank_test_score'] == n)[0][0]
    params = cv_results['params'][idx]
    mu     = cv_results['mean_test_score'][idx]
    std    = cv_results['std_test_score'][idx]
    return params, mu, std


def grid_search(emotions_tag, models, params, X_train, Y_train,
                random_seed=None, feature_list=None):
    """
    Run GridSearchCV for every model in *models* against X_train / Y_train.
    Returns a list of result dicts (one per model).
    """
    if feature_list is None:
        feature_list = range(X_train.shape[1])
    if random_seed is not None:
        np.random.seed(random_seed)

    results = []
    for key in models:
        gs = GridSearchCV(
            models[key], params[key],
            cv=2, scoring='roc_auc', n_jobs=-1, verbose=0
        ).fit(X_train, Y_train)

        best_params, mu, std = _n_best(gs.cv_results_)
        results.append({
            'emotion':    emotions_tag,
            'algorithm':  key,
            'params':     best_params,
            'mean_score': mu,
            'std_score':  std,
            'cv_result':  deepcopy(gs.cv_results_)
        })
    return results


# ── Training ────────────────────────────────────────────────────────────────────

def _extract_feature_importance(model, model_name, feature_names):
    """Unified feature importance extraction for EN, SVM, RF."""
    if model_name in ('EN', 'SVM') and hasattr(model, 'coef_'):
        weights = model.coef_.flatten()
        imp_df  = pd.DataFrame({'feature': feature_names, 'importance': weights})
        return weights, imp_df
    if model_name == 'RF' and hasattr(model, 'feature_importances_'):
        fi     = model.feature_importances_
        imp_df = pd.DataFrame({'feature': feature_names, 'importance': fi})
        return fi, imp_df
    return None, None


def _train_single(model_name, res_tab, X_train, Y_train, feature_names=None):
    """Train one model (used by Parallel)."""
    params = res_tab.loc[res_tab['algorithm'] == model_name, 'params'].iloc[0]
    model  = deepcopy(MODELS[model_name])
    model.set_params(**params)
    model.fit(X_train, Y_train)

    fi, fi_df = _extract_feature_importance(model, model_name, feature_names)
    return model_name, {
        'model':               model,
        'feature_importance':  fi,
        'feature_importance_df': fi_df,
    }


def train_models(res_tab, X_train, Y_train,
                 model_list=None, feature_names=None, n_jobs=1):
    """
    Train all (or a subset of) models using best hyperparameters from
    *res_tab* (output of grid_search).  Returns {model_name: {...}} dict.
    """
    if model_list is None:
        model_list = MODEL_NAMES

    trained = Parallel(n_jobs=n_jobs)(
        delayed(_train_single)(name, res_tab, X_train, Y_train, feature_names)
        for name in model_list
    )
    return dict(trained)


# ── Evaluation ──────────────────────────────────────────────────────────────────

def evaluate_models(trained_models, X_test, Y_test,
                    X_train=None, compute_shap=True):
    """
    Evaluate all trained models on X_test / Y_test.

    If X_train is provided and compute_shap=True, SHAP values are computed
    and stored in the output.

    Returns a dict: {model_name: metrics_dict}
    """
    report_dict = {}

    for key, entry in trained_models.items():
        model  = entry['model']
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None

        report = classification_report(Y_test, y_pred, output_dict=True)
        auc_score = roc_auc_score(Y_test, y_pred)

        record = {
            'Class 0 Precision': report['0']['precision'],
            'Class 0 Recall':    report['0']['recall'],
            'Class 0 F1-score':  report['0']['f1-score'],
            'Class 1 Precision': report['1']['precision'],
            'Class 1 Recall':    report['1']['recall'],
            'Class 1 F1-score':  report['1']['f1-score'],
            'auc':      auc_score,
            'accuracy': report['accuracy'],
            'y_prob':   y_prob,
            'y_pred':   y_pred,
            'y_test':   Y_test,
            'cm':       confusion_matrix(Y_test, y_pred),
        }

        # SHAP
        if compute_shap and X_train is not None:
            shap_vals, shap_exp = compute_shap_values(
                model, key, X_train, X_test, class_idx=1
            )
            record['shap_vals'] = shap_vals
            record['x_test']    = X_test

        report_dict[key] = record

    return report_dict

In [ ]:
print("Selected emotions:", select_emos)

## Make sure about the test and train dataset 

## 10. Main training loop (all emotions × all folds)

In [ ]:
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)

all_emo_perf   = []   # list[list[dict]]  — perf per emotion, per fold
all_models     = []   # list[list[dict]]  — trained models per emotion, per fold

for i_e, emotion in enumerate(select_emos):
    logging.info(f"Emotion {i_e+1}/{len(select_emos)}: {emotion}")

    # Build combined feature matrix: positive-labelled task rows + all baseline rows
    target_col = sel_target[:, i_e]
    pos_idx    = np.where(target_col == 1)[0]

    targets  = np.concatenate([
        target_col[pos_idx],
        np.zeros(len(base_feat_norm), dtype=int)
    ])
    features = pd.concat(
        [all_feat_norm.iloc[pos_idx], base_feat_norm], ignore_index=True
    )

    x_train, y_train, x_test, y_test = get_folds_data_fixed_train(
        n_split=N_FOLDS, feat_pd=features, target=targets.astype(int)
    )

    fold_perf   = []
    fold_models = []

    for fold, (X_tr, Y_tr, X_te, Y_te) in enumerate(
        zip(x_train, y_train, x_test, y_test)
    ):
        if len(set(Y_te)) < 2:
            logging.warning(f"  Fold {fold+1}: single class in test set — skipped.")
            continue

        logging.info(f"  Fold {fold+1}: HP tuning…")

        # Grid search → train
        gs_results = grid_search(
            emotion, MODELS, PARAM_GRIDS, X_tr, Y_tr,
            random_seed=RANDOM_SEED, feature_list=features_names
        )
        gs_df = pd.DataFrame(gs_results)

        trained = train_models(
            gs_df, X_tr, Y_tr,
            feature_names=features_names, n_jobs=1
        )
        perf = evaluate_models(trained, X_te, Y_te, X_train=X_tr)

        fold_perf.append(perf)
        fold_models.append(trained)

    all_emo_perf.append(fold_perf)
    all_models.append(fold_models)

logging.info("Training complete.")

## 11. Save model outputs

In [ ]:

def _save_pickle(obj, path):
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
    print(f"Saved → {path}")

_save_pickle(all_models,   SAVE_PATH + f"all_model_mt_emo_vs_rest_{timestamp}.pkl")
_save_pickle(all_emo_perf, SAVE_PATH + f"all_model_mt_modelsInfo_emo_vs_rest_{timestamp}.pkl")

## 12. Load saved outputs (run this cell to reload without retraining)

In [ ]:
if PID == 'emo05':
    LOAD_TIMESTAMP = '20260622'   # Emo 5
elif PID == 'emo06':
    LOAD_TIMESTAMP = '20260622'   # Emo 6 
elif PID == 'emo07':
    LOAD_TIMESTAMP = '20260622'   # Emo 7 
elif PID == 'emo01':
    LOAD_TIMESTAMP = '20260622'   # Emo 1
else:
    # Fallback to prevent a NameError
    raise ValueError(f"Unexpected or unconfigured PID: {PID}")

def _load_pickle(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

all_models   = _load_pickle(SAVE_PATH + f"all_model_mt_emo_vs_rest_{LOAD_TIMESTAMP}.pkl")
all_emo_perf = _load_pickle(SAVE_PATH + f"all_model_mt_modelsInfo_emo_vs_rest_{LOAD_TIMESTAMP}.pkl")

print(f"Loaded {len(all_models)} emotion model sets,  {len(all_emo_perf)} perf records.")

## 13. Build summary DataFrames from saved results

In [ ]:
def _unpack_results(nested_list, select_emos):
    """
    Flatten list[list[dict]] → pd.DataFrame with emotion / iter columns.
    Works for both all_emo_perf and all_models.
    """
    frames = []
    for emo, fold_list in zip(select_emos, nested_list):
        for i, fold_dict in enumerate(fold_list):
            df = pd.DataFrame.from_dict(fold_dict, orient='index')
            df['iter']       = f'iter_{i+1}'
            df['model_name'] = fold_dict.keys()
            df['emotion']    = emo
            frames.append(df)
    return pd.concat(frames, ignore_index=True)

## 14. SHAP aggregation across folds

In [ ]:
def build_shap_dict(report_df):
    """
    Aggregate SHAP values and X_test across all folds for each
    (model, emotion) pair.
    Returns {(model, emotion): {'shap_vals': ndarray, 'x_test': ndarray}}
    """
    results = {}
    for (model, emotion), grp in report_df.groupby(['model_name', 'emotion'], sort=False):
        shap_all = np.vstack(grp['shap_vals'].tolist())
        x_all    = np.vstack(grp['x_test'].tolist())
        results[(model, emotion)] = {'shap_vals': shap_all, 'x_test': x_all}
    return results


results_shap = build_shap_dict(
    report_df.dropna(subset=['shap_vals', 'x_test'])
)
print(f"SHAP dict has {len(results_shap)} (model, emotion) entries.")

## 15. Performance visualisation

In [ ]:


# 15 colorblind-friendly, visually distinct colors
# Sources: Bang Wong (Nature Methods), Paul Tol's qualitative palette,
# supplemented with tested additions for ≥12 categories.
EMOTION_PALETTE = [
    "#332288",  # indigo
    "#88CCEE",  # sky blue
    "#44AA99",  # teal
    "#117733",  # green
    "#999933",  # olive
    "#DDCC77",  # sand
    "#CC6677",  # rose
    "#882255",  # wine
    "#AA4499",  # purple
    "#444444",  # light gray
    "#E69F00",  # amber
    "#56B4E9",  # light blue
    "#D55E00",  # vermilion
    "#0072B2",  # deep blue
    "#F0E442",  # yellow
]


# MASTER LIST: This strictly locks each emotion to the corresponding color above.
ALL_EMOTIONS = [
    "Calmness", "Entrancement", "Confusion", "Boredom", 
    "Interest", "Satisfaction", "Relief", "Excitement", 
    "Awkwardness", "Amusement", "Adoration", "Joy", 
    "Admiration", "Awe", "AestheticAppreciation"
]


def plot_performance_cl(report_df, select_emos, pid, save_path=None):
    """2×2 bar-chart grid: AUC / Accuracy / Class-0 F1 / Class-1 F1."""
    metrics = [
        ('auc',              'AUC — ' + pid),
        ('accuracy',         'Accuracy'),
        ('Class 0 F1-score', 'Class 0 F1-score'),
        ('Class 1 F1-score', 'Class 1 F1-score'),
    ]

    # Map each emotion to a fixed color across all subplots

    

    master_palette = {emo: color for emo, color in zip(ALL_EMOTIONS, EMOTION_PALETTE)}
    
    # 2. Apply visibility fix to the master palette
    if 'Amusement' in master_palette:
        master_palette['Amusement'] = '#444444' # Visibility fix

    # 3. Filter the master palette to only include the emotions currently being plotted.
    # This guarantees the colors will NEVER shift, even if some are missing.
    palette = {emo: master_palette[emo] for emo in select_emos}

  

    fig, axes = plt.subplots(2, 2, figsize=(20, 10))
    axes = axes.flatten()

    for ax, (metric, title) in zip(axes, metrics):
        mean_df = (
            report_df
            .groupby(['model_name', 'emotion', 'iter'], sort=False)[metric]
            .mean().reset_index()
        )
        sns.barplot(
            data=mean_df, x='model_name', y=metric,
            hue='emotion', hue_order=select_emos,
            edgecolor='white', linewidth=0.5,
            ax=ax, palette=palette,
        )
        ax.set_title(title, fontweight='bold')
        ax.axhline(0.5, color='black', linestyle='--', linewidth=1.5)
        ax.set_ylim(0.2, 1)
        if metric != 'accuracy':
            ax.get_legend().remove()
        else:
            ax.legend(
                bbox_to_anchor=(1.05, 1), loc='upper left',fontsize = 15,
                frameon=True, edgecolor='#cccccc',
            )

    fig.tight_layout()
    
    if save_path:
        timestamp = datetime.now().strftime("%Y%m%d")
        fig.savefig(save_path + f"All_model_perform_mt_eachEmoVsBase_{pid}_{timestamp}.svg", dpi=300)
    plt.show()



In [ ]:
plot_performance_cl(report_df, select_emos, PID)
# plot_performance_cl(report_df, select_emos, PID, save_path=FIG_PATH)

## 16. ROC curves with CI

In [ ]:
def plot_roc_with_ci(df_toplot, ax=None):
    """
    Plot mean ROC ± SD for each model in df_toplot.
    df_toplot must have columns: y_test, y_prob, model_name, iter.
    Returns (roc_df, ax).
    """
    common_fpr = np.linspace(0, 1, 100)
    roc_data   = []

    for _, row in df_toplot.iterrows():
        fpr, tpr, thresh = roc_curve(row['y_test'], row['y_prob'])
        tpr_i = np.interp(common_fpr, fpr, tpr)
        roc_auc = auc(fpr, tpr)
        # for f, t,th in zip(common_fpr, tpr_i,thresh):
        #     roc_data.append({
        #         'Model': row['model_name'],
        #         'FPR': f, 'TPR': t, 'Thresh': th, 
        #         'AUC': roc_auc, 'Iter': row['iter']
        #     })

        for f, t  in zip(common_fpr, tpr_i):
            roc_data.append({
                'Model': row['model_name'],
                'FPR': f, 'TPR': t, 
                'AUC': roc_auc, 'Iter': row['iter']
        })   

    roc_df = pd.DataFrame(roc_data)

    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))

    sns.lineplot(
        x='FPR', y='TPR', hue='Model',
        data=roc_df, palette='Set1',
        lw=2, ci='sd', estimator='mean',
        legend='brief', ax=ax
    )

    # Update legend with mean AUC
    handles, labels = ax.get_legend_handles_labels()
    for i, model in enumerate(roc_df['Model'].unique()):
        mean_auc = roc_df[roc_df['Model'] == model]['AUC'].mean()
        labels[i] = f"{model} (AUC={mean_auc:.2f})"
    ax.legend(handles, labels, loc='lower right', fontsize=14)

    ax.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--')
    ax.set(xlim=[0, 1], ylim=[0, 1.05],
           xlabel='False Positive Rate',
           ylabel='True Positive Rate')
    ax.grid(True, linestyle='--', alpha=0.5)
    return roc_df, ax


# Plot ROC for all emotions
n_emos = len(select_emos)
n_cols = 5
n_rows = -(-n_emos // n_cols)   # ceiling division
fig, axes = plt.subplots(n_rows, n_cols, figsize=(7 * n_cols, 6 * n_rows))
axes = axes.flatten()

for i, emotion in enumerate(select_emos):
    sel = report_df[report_df['emotion'] == emotion].reset_index()
    _, ax = plot_roc_with_ci(sel, ax=axes[i])
    axes[i].set_title(emotion, fontsize=14)

for ax in axes[n_emos:]:
    ax.set_visible(False)

fig.tight_layout()
timestamp = datetime.now().strftime("%Y%m%d")
fig.savefig(FIG_PATH + f"All_model_allemo_vsRest_ROCSmooth_mt{timestamp}.svg",
            dpi=300, bbox_inches='tight')
plt.show()

## Save the performance metrics

In [ ]:

EN_PERF_STORE = os.path.join(DATA_ROOT, "EN_performance_metrics_allemo_allpid.pkl")
METRIC_COLS   = ['auc', 'accuracy', 'Class 0 F1-score', 'Class 1 F1-score']


def save_en_metrics(report_df, pid, path=EN_PERF_STORE):
    """Compute EN mean±SEM per emotion and upsert under `pid` in a pickle dict."""
    if not isinstance(pid, str) or not pid:
        raise ValueError(f"pid must be a non-empty string, got {pid!r}")

    en = report_df[report_df['model_name'] == 'EN']
    if en.empty:
        raise ValueError("No EN rows in report_df — refusing to save (stale report_df?).")

    per_iter = (en.groupby(['emotion', 'iter'], sort=False)[METRIC_COLS]
                  .mean().reset_index())
    summary = (per_iter.groupby('emotion', sort=False)[METRIC_COLS]
                       .agg(['mean', 'sem']))
    summary.columns = [f"{m}_{s}" for m, s in summary.columns]
    summary = summary.reset_index()

    store = {}
    if os.path.exists(path):
        with open(path, "rb") as f:
            store = pickle.load(f)

    store[pid] = summary                       # replace just this PID
    with open(path, "wb") as f:
        pickle.dump(store, f)

    print(f"Saved EN metrics for {pid!r} → {path}")
    print(f"PIDs now in store: {list(store)}")
    return summary


def load_en_metrics(pid=None, path=EN_PERF_STORE):
    """Return the whole {PID: DataFrame} dict, or one PID's DataFrame.
    A missing PID raises KeyError — never returns the wrong one."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"No metrics store at {path}")
    with open(path, "rb") as f:
        store = pickle.load(f)
    if pid is None:
        return store
    if pid not in store:
        raise KeyError(f"PID {pid!r} not in store. Available: {list(store)}")
    return store[pid]


def load_en_metrics_all(path=EN_PERF_STORE):
    """Flatten the store into one tidy DataFrame with a PID column."""
    store = load_en_metrics(path=path)
    frames = []
    for pid, df in store.items():
        d = df.copy()
        d.insert(0, 'PID', pid)
        frames.append(d)
    return pd.concat(frames, ignore_index=True)




In [ ]:
save_en_metrics(report_df, PID)


## Load the performance metric

In [ ]:
perf = load_en_metrics(PID)
perf.head()
load_en_metrics_all()

## Plot all EN and emotions in one panel 

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from sklearn.metrics import roc_curve, auc
from datetime import datetime

def plot_en_roc_minimal_clipped(report_df, select_emos, pid, save_path=None):
    # 1. Accessible Publication Palette
    # custom_hex = [
    #     "#264653", "#2A9D8F", "#E9C46A", "#F4A261", "#E76F51", 
    #     "#6D597A", "#B56576", "#EAAC8B", "#1D3557", "#2D6A4F", 
    #     "#457B9D", "#A8DADC", "#525B76", "#D4A373", "#603808" 
    # ]
    ALL_EMOTIONS = [
    "Calmness", "Entrancement", "Confusion", "Boredom", 
    "Interest", "Satisfaction", "Relief", "Excitement", 
    "Awkwardness", "Amusement", "Adoration", "Joy", 
    "Admiration", "Awe", "AestheticAppreciation"
    ]
    # ------------colors from cl----------------
    custom_hex  = [
        "#332288",  # indigo
        "#88CCEE",  # sky blue
        "#44AA99",  # teal
        "#117733",  # green
        "#999933",  # olive
        "#DDCC77",  # sand
        "#CC6677",  # rose
        "#882255",  # wine
        "#AA4499",  # purple
        "#444444",  # dark gray (visibility fix already applied)
        "#E69F00",  # amber
        "#56B4E9",  # light blue
        "#D55E00",  # vermilion
        "#0072B2",  # deep blue
        "#F0E442",  # yellow
    ]


    
     # 1. Create a MASTER mapping of all 15 emotions to their fixed colors
    master_palette = {emo: color for emo, color in zip(ALL_EMOTIONS, custom_hex)}
    
    # 2. Filter the master palette to only include the emotions currently being plotted.
    # This guarantees the colors will NEVER shift, even if some are missing.
    palette = {emo: master_palette[emo] for emo in select_emos}


    df_en = report_df[report_df['model_name'] == 'EN'].copy()
    common_fpr = np.linspace(0, 1, 100)
    roc_data = []

    for emotion in select_emos:
        emo_subset = df_en[df_en['emotion'] == emotion]
        for _, row in emo_subset.iterrows():
            fpr, tpr, _ = roc_curve(row['y_test'], row['y_prob'])
            tpr_interp = np.interp(common_fpr, fpr, tpr)
            roc_auc = auc(fpr, tpr)
            for f, t in zip(common_fpr, tpr_interp):
                roc_data.append({'Emotion': emotion, 'FPR': f, 'TPR': t, 'AUC': roc_auc})

    roc_df = pd.DataFrame(roc_data)

    # 2. Visualization Setup
    sns.set_style("whitegrid", {'axes.grid': True, 'grid.linestyle': '--', 'grid.alpha': 0.4})
    fig, ax = plt.subplots(figsize=(12, 10))

    # Force the plotting to clip exactly at the axes edges
    line_plot = sns.lineplot(
        x='FPR', y='TPR', hue='Emotion',
        data=roc_df, hue_order=select_emos,
        # palette=custom_hex[:len(select_emos)],
        palette=palette,  # Use the fixed palette
        lw=2.5, errorbar='sd', estimator='mean',
        ax=ax
    )

    # ── THE FIX: CLIP EVERYTHING TO [0, 1] ──────────────────────────────────
    # This prevents the shaded SD bands from bleeding outside the frame
    for artist in ax.get_lines() + ax.collections:
        artist.set_clip_path(ax.patch)

    # 3. THE "ONLY ONE ZERO" FORMATTER
    def clean_format(x, pos):
        if x == 0: return "0"
        if x == 1: return "1"
        return f"{x:.1f}".replace("0.", ".")

    formatter = FuncFormatter(clean_format)
    ax.xaxis.set_major_formatter(formatter)
    ax.yaxis.set_major_formatter(formatter)
    
    ax.set_xticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])

    # ── THE FIX: INCREASE TICK FONT SIZE ──────────────────────────────────
    ax.tick_params(axis='both', which='major', labelsize=16)

    # 4. Legend & Baseline
    new_labels = []
    for emotion in select_emos:
        mean_auc = roc_df[roc_df['Emotion'] == emotion]['AUC'].mean()
        new_labels.append(f"{emotion} ({mean_auc:.2f})")

    leg = ax.legend(
        ax.get_legend_handles_labels()[0], new_labels, 
        loc='lower right', ncol=2, fontsize=15, 
        title="Emotion (Mean AUC)", frameon=True, facecolor='white',
        edgecolor='#CCCCCC'
    )
    plt.setp(leg.get_title(), fontsize='15', fontweight='bold')

    # Chance level (Dashed Black)
    ax.plot([0, 1], [0, 1], color='black', lw=1.5, linestyle='--', alpha=0.6, zorder=1)
    
    # 5. Final Styling & Strict Limits
    ax.set_title(f"ROC Performance: Elastic Net — {pid}", fontsize=18, fontweight='bold', pad=25)
    ax.set_xlabel('False Positive Rate', fontsize=14, fontweight='bold')
    ax.set_ylabel('True Positive Rate', fontsize=14, fontweight='bold')
    
    # Set limits to EXACTLY 0 and 1 to "box in" the data
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    
    # Remove the trim=True to keep the axis lines meeting at the corner
    sns.despine(left=False, bottom=False)
    
    plt.tight_layout()

    if save_path:
        timestamp = datetime.now().strftime("%Y%m%d")
        fig.savefig(save_path + f"EN_ROC_AllEmoOverlayed_{pid}_{timestamp}.svg", dpi=300)
    plt.show()

# plot_en_roc_minimal_clipped(report_df, select_emos, PID, save_path=FIG_PATH)
plot_en_roc_minimal_clipped(report_df, select_emos,PID)

##  Stable feature extraction (Elastic Net)

In [ ]:
def extract_stable_features(model_df, feature_names, model_name='EN', threshold=0.8):
    # """
    # For each emotion, select features that appear (non-zero weight) in at least
    # *threshold* fraction of cross-validation runs.

    # EN weights are max-abs scaled per run before aggregation.
    # RF importances are used as-is.

    # Returns a wide DataFrame (one quadruple of columns per emotion):
    #   <emotion>_Stable_Feature, <emotion>_Mean_Importance, <emotion>_SEM,
    #   <emotion>_Frequency
    # """
    target_df = model_df[model_df['model_name'] == model_name].copy()
    if target_df.empty:
        print(f"No '{model_name}' entries found.")
        return pd.DataFrame()

    all_results = []

    for emotion in target_df['emotion'].unique():
        subset   = target_df[target_df['emotion'] == emotion]
        num_runs = len(subset)
        matrix   = np.array(subset['feature_importance'].tolist(), dtype=float)

        # Per-run max-abs scaling for EN
        # if model_name == 'EN':
        #     row_max = np.max(np.abs(matrix), axis=1, keepdims=True)
        #     row_max[row_max == 0] = 1.0
        #     matrix = matrix / row_max

        is_selected     = np.abs(matrix) > 1e-9
        sel_counts      = is_selected.sum(axis=0)

        threshold_count = int(threshold * num_runs)
        stable_idx = np.where(sel_counts >= threshold_count)[0]
        if len(stable_idx) == 0:
            print(f"  [{emotion}] No features reached {threshold*100:.0f}% threshold.")
            continue

        stable_mat   = matrix[:, stable_idx]
        stable_names = [feature_names[i] for i in stable_idx]
        mean_vals    = stable_mat.mean(axis=0)
        sem_vals     = stats.sem(stable_mat, axis=0)
        freq_vals    = sel_counts[stable_idx] / num_runs

        temp = pd.DataFrame({
            f'{emotion}_Stable_Feature':   stable_names,
            f'{emotion}_Mean_Importance':  mean_vals,
            f'{emotion}_SEM':              sem_vals,
            f'{emotion}_Frequency':        freq_vals,
        })
        abs_mean_col = f'{emotion}_Mean_Importance'
        temp['_abs'] = temp[abs_mean_col].abs()
        temp = temp.sort_values(['_abs'], ascending=False).drop(columns='_abs')
        temp = temp.reset_index(drop=True)
        all_results.append(temp)

    return pd.concat(all_results, axis=1) if all_results else pd.DataFrame()


en_stable_df = extract_stable_features(
    models_df, features_names, model_name='EN', threshold=0.9
)
en_stable_df.head(5)

##  Common / specific / distinct feature sets

In [ ]:
def get_feature_sets(stable_df):
    """
    Derive three complementary feature sets from a stable-feature DataFrame:
      - common:   present in every emotion's stable list
      - specific: stable for each emotion but not universally common
      - distinct: stable for exactly one emotion

    Returns (common_list, specific_df, distinct_df)
    """
    feat_cols = [c for c in stable_df.columns if '_Stable_Feature' in c]
    imp_cols  = [c for c in stable_df.columns if '_Mean_Importance' in c]

    sets       = {c: set(stable_df[c].dropna()) for c in feat_cols}
    all_sets   = list(sets.values())
    common_set = set.intersection(*all_sets)
    common_list = common_set
    print(common_list)

    union_all  = set.union(*all_sets)

    # ── Common ───────────────────────────────────────────────────────────────
    # common_list = sorted(common_set)
    print(f"{len(common_list)} common features.")

    # ── Specific (present, but not universal) ─────────────────────────────────
    spec_frames = []
    for fc, ic in zip(feat_cols, imp_cols):
        emo  = fc.replace('_Stable_Feature', '')
        rows = stable_df[[fc, ic]].dropna()
        rows = rows[~rows[fc].isin(common_set)].copy()
        rows.columns = [f'{emo}_Specific_Feature', f'{emo}_Importance']
        spec_frames.append(rows.reset_index(drop=True))
    specific_df = pd.concat(spec_frames, axis=1)
    # print(common_list)


    # ── Distinct (present in exactly one emotion) ─────────────────────────────
    dist_frames = []
    for fc, ic in zip(feat_cols, imp_cols):
        emo          = fc.replace('_Stable_Feature', '')
        current_set  = sets[fc]
        others_union = set.union(*(s for c, s in sets.items() if c != fc))
        distinct_set = current_set - others_union

        rows = stable_df.loc[stable_df[fc].isin(distinct_set), [fc, ic]].copy()
        rows.columns = [f'{emo}_Distinct_Feature', f'{emo}_Importance']
        rows = rows.sort_values(f'{emo}_Importance', key=abs, ascending=False)
        dist_frames.append(rows.reset_index(drop=True))
    distinct_df = pd.concat(dist_frames, axis=1)
    # print(common_list)


    return common_list, specific_df, distinct_df


common_features, specific_stable_df, distinct_df = get_feature_sets(en_stable_df)
# print(distinct_df.head(5))

# Extract the features and remove the tail. when the CUMsum reaches 90% 

In [ ]:
import pandas as pd
import numpy as np

def remove_tail_by_plateau(en_stable_df, total_importance_threshold=0.90):
    """
    Identifies the 'head' features (most important) and removes the 'tail' 
    features based on the cumulative distribution plateau.
    
    Returns a new wide-format DataFrame containing only the significant features.
    """
    emo_cols = [c for c in en_stable_df.columns if '_Stable_Feature' in c]
    emotions = [c.replace('_Stable_Feature', '') for c in emo_cols]

    head_results = []

    for emo in emotions:
        cols = [f'{emo}_Stable_Feature', f'{emo}_Mean_Importance', 
                f'{emo}_SEM', f'{emo}_Frequency']
        
        # 1. Extract and clean data for this specific emotion
        subset = en_stable_df[cols].dropna().copy()
        if subset.empty: 
            continue

        # 2. Sort by Absolute Importance (Descending)
        subset['abs_val'] = subset[f'{emo}_Mean_Importance'].abs()
        subset = subset.sort_values('abs_val', ascending=False)

        # 3. Calculate Cumulative Percentage
        total_sum = subset['abs_val'].sum()
        subset['cum_perc'] = subset['abs_val'].cumsum() / total_sum

        # 4. Filter: Keep only the 'Head' 
        # We keep everything UNTIL we hit the threshold.
        # This effectively "removes" the tail plateau.
        head_subset = subset[subset['cum_perc'] <= total_importance_threshold].copy()
        
        # Optional: Ensure at least the top 1 feature is always kept, 
        # even if it alone exceeds the threshold.
        if head_subset.empty and not subset.empty:
            head_subset = subset.head(1).copy()

        # 5. Clean up temporary columns and reset index
        head_subset = head_subset[cols].reset_index(drop=True)
        head_results.append(head_subset)

    # 6. Concatenate into a new wide-format DataFrame
    # Features removed from the tail will now be NaNs at the bottom of the columns
    en_stable_head_df = pd.concat(head_results, axis=1)
    
    return en_stable_head_df

# Example Usage:
# This creates a version of your stable features containing only the top 90% of importance
en_stable_trimmed = remove_tail_by_plateau(en_stable_df, total_importance_threshold=0.90)

## SAVE en_Stable_df and trimmed

In [ ]:
print("length en_stable_df", len(en_stable_df ))
print("length en_stable_trimmed", len(en_stable_trimmed))


# One shared store for all PIDs, living at DATA_ROOT (not inside a PID folder)
STORE_PATH = os.path.join(DATA_ROOT, "en_stable_store.pkl")

# Load what's already there, or start fresh
if os.path.exists(STORE_PATH):
    with open(STORE_PATH, "rb") as f:
        en_stable_store = pickle.load(f)
else:
    en_stable_store = {}

# Add / overwrite the current participant
en_stable_store[PID] = {
    "en_stable_df":      en_stable_df.copy(),
    "en_stable_trimmed": en_stable_trimmed.copy(),
}

with open(STORE_PATH, "wb") as f:
    pickle.dump(en_stable_store, f)

print(f"Saved {PID} → {STORE_PATH}")
print(f"PIDs in store: {list(en_stable_store)}")


## Load the en-Stable_Df 

In [ ]:
import os
import pickle

STORE_PATH = os.path.join(DATA_ROOT, "en_stable_store.pkl")
with open(STORE_PATH, "rb") as f:
    en_stable_store = pickle.load(f)

# Access a participant
en_stable_df      = en_stable_store[PID]["en_stable_df"]
en_stable_trimmed = en_stable_store[PID]["en_stable_trimmed"]

In [ ]:

def plot_all_stable_emotions_compact(df, n_rows, n_cols, figsize=(22, 40)):
    """
    Plots stable features in a grid defined by n_rows and n_cols.
    Optimized for tight vertical spacing and readable titles.
    """
    # 1. Identify all emotions in the dataframe
    emo_cols = [c for c in df.columns if '_Stable_Feature' in c]
    all_emotions = [c.replace('_Stable_Feature', '') for c in emo_cols]

    # 2. Set up the grid using provided input dimensions
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    
    # Handle the case where n_rows * n_cols == 1
    if n_rows * n_cols == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    en_color = '#4a82d6' 

    for i, emotion in enumerate(all_emotions):
        if i >= len(axes):
            break
            
        ax = axes[i]
        
        # 3. Extract columns
        cols = [f'{emotion}_Stable_Feature', f'{emotion}_Mean_Importance', f'{emotion}_SEM']
        existing_cols = [c for c in cols if c in df.columns]
        
        if len(existing_cols) < 3:
            ax.axis('off')
            continue
            
        data = df[existing_cols].dropna().copy()
        
        if data.empty:
            ax.set_title(f"{emotion} (No Features)", fontsize=10)
            ax.axis('off')
            continue
            
        data.columns = ['feature', 'mean', 'sem']
        
        # 4. Sorting
        data['abs_mean'] = data['mean'].abs()
        data = data.sort_values('abs_mean', ascending=True)
        
        # 5. Dynamic sizing for labels
        num_features = len(data)
        font_size = 18 if num_features < 40 else 16
        
        # 6. Plotting
        y_pos = np.arange(num_features)
        ax.barh(y_pos, data['mean'], color=en_color, align='center', alpha=0.9)
        
        # 7. Add Error Bars
        ax.errorbar(data['mean'], y_pos, xerr=data['sem'], fmt='none', 
                    color='black', capsize=2, elinewidth=0.6)
        
        # 8. Axis Formatting
        ax.set_yticks(y_pos)
        clean_labels = [str(f).replace('feat_', '') for f in data['feature']]
        ax.set_yticklabels(clean_labels, fontsize=font_size)
        
        # Compact title and padding
        ax.set_title(f" {emotion} (n={num_features})", 
                     fontsize=22, fontweight='bold', pad=4)
        
        ax.set_xlabel('Mean Importance', fontsize=18)
        ax.axvline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.3)
        
        sns.despine(ax=ax)

    # Hide any extra subplots
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    # ─── THE SPACING FIX ───
    # h_pad=0.2 removes the dead air between the rows
    plt.tight_layout(h_pad=2, w_pad=1.5)
    
    return fig

# --- Execution ---
# Now you can enter the rows and columns as inputs here:
# plt.ioff()  # Disable interactive mode for better layout control
fig = plot_all_stable_emotions_compact(en_stable_trimmed, n_rows=5, n_cols=3, figsize=(42,70))
plt.show()


fig.savefig(FIG_PATH + f"EN_Significant_Features_allEmot_{timestamp}_{PID}.svg",
            dpi=300, bbox_inches='tight')
fig.savefig(FIG_PATH + f"EN_Significant_Features_allEmot_{timestamp}_{PID}.pdf",
            dpi=300, bbox_inches='tight')



# plot stable features for one emotion at a time 

In [ ]:
def plot_stable_emotions(df, target_emotion=None, n_rows=None, n_cols=None, figsize=None):
    """
    Plots stable features. Can plot all emotions in a grid or a single target emotion.
    """
    # 1. Determine mode: Single Emotion vs All Emotions Grid
    if target_emotion is not None:
        all_emotions = [target_emotion]
        n_rows, n_cols = 1, 1
        if figsize is None:
            # Smart default size for a single panel
            figsize = (10, 8)
    else:
        if n_rows is None or n_cols is None:
            raise ValueError("You must specify n_rows and n_cols when plotting all emotions.")
        emo_cols = [c for c in df.columns if '_Stable_Feature' in c]
        all_emotions = [c.replace('_Stable_Feature', '') for c in emo_cols]
        if figsize is None:
            figsize = (22, 40)

    # 2. Set up the grid
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    
    if n_rows * n_cols == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    en_color = '#4a82d6' 

    for i, emotion in enumerate(all_emotions):
        if i >= len(axes):
            break
            
        ax = axes[i]
        
        # 3. Extract columns
        cols = [f'{emotion}_Stable_Feature', f'{emotion}_Mean_Importance', f'{emotion}_SEM']
        existing_cols = [c for c in cols if c in df.columns]
        
        if len(existing_cols) < 3:
            ax.axis('off')
            continue
            
        data = df[existing_cols].dropna().copy()
        
        if data.empty:
            ax.set_title(f"{emotion} (No Features)", fontsize=12, fontweight='bold')
            ax.axis('off')
            continue
            
        data.columns = ['feature', 'mean', 'sem']
        
        # 4. Sorting
        data['abs_mean'] = data['mean'].abs()
        data = data.sort_values('abs_mean', ascending=True)
        
        # 5. Dynamic sizing for labels
        num_features = len(data)
        font_size = 13 if num_features < 40 else 11
        
        # Adjust single plot figure height dynamically if it's a massive feature list
        if target_emotion is not None and num_features > 30 and figsize == (10, 8):
            fig.set_figheight(num_features * 0.25)
        
        # 6. Plotting
        y_pos = np.arange(num_features)
        ax.barh(y_pos, data['mean'], color=en_color, align='center', alpha=0.9, edgecolor='none')
        
        # 7. Add Error Bars
        ax.errorbar(data['mean'], y_pos, xerr=data['sem'], fmt='none', 
                    color='black', capsize=2, elinewidth=0.6)
        
        # 8. Axis Formatting
        ax.set_yticks(y_pos)
        clean_labels = [str(f).replace('feat_', '') for f in data['feature']]
        ax.set_yticklabels(clean_labels, fontsize=font_size)
        
        ax.set_title(f"{emotion} (n={num_features})", fontsize=14, fontweight='bold', pad=8)
        ax.set_xlabel('Mean Importance', fontsize=12, fontweight='bold')
        ax.axvline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.3)
        
        sns.despine(ax=ax)

    # Hide any extra subplots if in grid mode
    if target_emotion is None:
        for j in range(i + 1, len(axes)):
            axes[j].set_visible(False)

    plt.tight_layout(h_pad=0.2, w_pad=1.5)
    return fig

In [ ]:
target_emo = "Joy"

fig = plot_stable_emotions(en_stable_trimmed, target_emotion=target_emo, figsize=(10, 8))
plt.show()

# Saves as: "EN_Significant_Features_Joy.svg"
save_filename = f"EN_Significant_Features_{target_emo}_{timestamp}_{PID}.svg"
fig.savefig(os.path.join(FIG_PATH, save_filename), dpi=300, bbox_inches='tight')

# plot feature value correlation from the en_Stable 

In [ ]:

def compute_and_cluster_emotion_correlation_V4(task_df, base_df, target, stable_df, 
                                               all_emotion_names, emos_to_cluster=None, 
                                               tick_font_size=12, fig_path=None):
    """
    Computes correlation matrices for specific emotions, finds optimal clustering 
    via Silhouette scores, and saves publication-quality SVG clustermaps.
    """
    if emos_to_cluster is None:
        emos_to_cluster = all_emotion_names
    
    results = {}
    all_corr_matrices = {}
    timestamp = datetime.now().strftime("%Y%m%d")

    for emotion in emos_to_cluster:
        if emotion not in all_emotion_names:
            continue
        
        feature_col_name = f"{emotion}_Stable_Feature"
        current_feature_cols = stable_df[feature_col_name].dropna().tolist()
        
        if not current_feature_cols or len(current_feature_cols) < 2:
            continue

        # Prepare data
        i_e = all_emotion_names.index(emotion)
        pos_idx = np.where(target[:, i_e] == 1)[0]
        emo_task_rows = task_df.iloc[pos_idx][current_feature_cols]
        combined = pd.concat([emo_task_rows, base_df[current_feature_cols]], axis=0)
        
        # Compute Correlation
        corr_matrix = combined.corr(method='pearson').fillna(0)
        all_corr_matrices[emotion] = corr_matrix
        
        # 1. Perform Linkage
        Z = linkage(corr_matrix, method='ward', metric='euclidean', optimal_ordering=True)
        
        # 2. Silhouette Analysis
        best_k, max_silhouette = 2, -1
        max_test_k = min(15, len(current_feature_cols))
        for k in range(2, max_test_k):
            labels = fcluster(Z, t=k, criterion='maxclust')
            score = silhouette_score(corr_matrix, labels, metric='euclidean')
            if score > max_silhouette:
                max_silhouette, best_k = score, k

        # 3. Extract clusters
        optimal_labels = fcluster(Z, t=best_k, criterion='maxclust')
        feature_list = corr_matrix.index.tolist()
        cluster_mapping = {f"Cluster_{c}": [feature_list[i] for i, l in enumerate(optimal_labels) if l == c] 
                           for c in range(1, best_k + 1)}
        
        results[emotion] = {
            'optimal_k': best_k,
            'silhouette_score': max_silhouette,
            'clusters': cluster_mapping,
            'full_df': pd.DataFrame({'feature': feature_list, 'cluster': optimal_labels})
        }

        # 4. Visualization
        # Using the matte/lively palette style without edges for publication
        g = sns.clustermap(
            corr_matrix, 
            row_linkage=Z, 
            col_linkage=Z, 
            cmap='Spectral_r', 
            center=0, 
            vmin=-1, vmax=1, 
            figsize=(14, 12),
            xticklabels=True, 
            yticklabels=True,
            dendrogram_ratio=0.12,
            # Kept the high-left position you preferred
            cbar_pos=(0.02, 0.85, 0.03, 0.12) 
        )

        # Applying the increased font sizes
        plt.setp(g.ax_heatmap.get_xticklabels(), fontsize=tick_font_size, rotation=45, ha='right')
        plt.setp(g.ax_heatmap.get_yticklabels(), fontsize=tick_font_size, rotation=0)
        
        g.fig.suptitle(
            f'Feature Correlation: {emotion} ({len(current_feature_cols)} Features)\n'
            f'Optimal Clusters: {best_k} (Silhouette Score: {max_silhouette:.2f})', 
            fontsize=16, y=1.03, fontweight='bold'
        )

        # --- AUTO-SAVE LOGIC ---
        if fig_path:
            if not os.path.exists(fig_path):
                os.makedirs(fig_path)
            
            save_name = f"{emotion}_Clustermap_{timestamp}.svg"
            full_save_path = os.path.join(fig_path, save_name)
            
            # Save before showing; bbox_inches='tight' is key for SVG labels
            g.savefig(full_save_path, format='svg', dpi=300, bbox_inches='tight')
            print(f"Successfully saved {emotion} to: {full_save_path}")

        plt.show()

    return results, all_corr_matrices

In [ ]:

# # Assuming FIG_PATH is defined earlier in your script
output, matrices = compute_and_cluster_emotion_correlation_V4(
   all_feat_norm, base_feat_norm, sel_target, en_stable_trimmed, 
    all_emotion_names=select_emos,
    emos_to_cluster=select_emos, 
    fig_path=FIG_PATH
)




## save the output and matrices from feature correlation 

In [ ]:


# Shared store for all PIDs, at DATA_ROOT level
CLUSTER_STORE_PATH = os.path.join(DATA_ROOT, "cluster_output_store.pkl")

if os.path.exists(CLUSTER_STORE_PATH):
    with open(CLUSTER_STORE_PATH, "rb") as f:
        cluster_store = pickle.load(f)
else:
    cluster_store = {}

cluster_store[PID] = {
    "output":   output,      # per-emotion: optimal_k, silhouette_score, clusters, full_df
    "matrices": matrices,    # per-emotion correlation DataFrames
}

with open(CLUSTER_STORE_PATH, "wb") as f:
    pickle.dump(cluster_store, f)

print(f"Saved clustering for {PID} → {CLUSTER_STORE_PATH}")
print(f"PIDs in store: {list(cluster_store)}")
print(f"Emotions for {PID}: {list(output)}")

## Load the Clustering result for feature correlation

In [ ]:
import os
import pickle

CLUSTER_STORE_PATH = os.path.join(DATA_ROOT, "cluster_output_store.pkl")
with open(CLUSTER_STORE_PATH, "rb") as f:
    cluster_store = pickle.load(f)

output   = cluster_store[PID]["output"]
matrices = cluster_store[PID]["matrices"]

# e.g. pull one emotion's pieces back out
adoration = output["Adoration"]
print(adoration["optimal_k"], adoration["silhouette_score"])
adoration["full_df"].head()
matrices["Adoration"].head()

## plot features with cluster coloring

In [ ]:
from matplotlib.patches import Patch
def plot_all_stable_emotions_compact_colorV2(df, cluster_dict, n_rows, n_cols, figsize=(22, 40)):
    """
    Plots stable features in a grid.
    Colors bars based on cluster assignment and adds a cluster legend to each subpanel.
    """
    emo_cols = [c for c in df.columns if '_Stable_Feature' in c]
    all_emotions = [c.replace('_Stable_Feature', '') for c in emo_cols]

    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    
    if n_rows * n_cols == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    # Define a distinct color palette
    palette = sns.color_palette("tab20", 20)
    default_color = '#4a82d6' 

    for i, emotion in enumerate(all_emotions):
        if i >= len(axes):
            break
            
        ax = axes[i]
        cols = [f'{emotion}_Stable_Feature', f'{emotion}_Mean_Importance', f'{emotion}_SEM']
        existing_cols = [c for c in cols if c in df.columns]
        
        if len(existing_cols) < 3:
            ax.axis('off')
            continue
            
        data = df[existing_cols].dropna().copy()
        if data.empty:
            ax.axis('off')
            continue
            
        data.columns = ['feature', 'mean', 'sem']
        data['abs_mean'] = data['mean'].abs()
        data = data.sort_values('abs_mean', ascending=True)
        
        # --- CLUSTER COLORING & LEGEND LOGIC ---
        emo_clusters = cluster_dict.get(emotion, {}).get('clusters', {})
        
        feature_to_color = {}
        feature_to_cluster_name = {}
        
        # Keep track of which clusters are present in this specific emotion's data
        present_clusters = {} 

        for cluster_name, features in emo_clusters.items():
            try:
                # Extract number for consistent coloring across subplots
                cluster_num = int(cluster_name.split('_')[1])
                color = palette[(cluster_num - 1) % len(palette)]
            except:
                color = default_color
            
            for feat in features:
                feature_to_color[feat] = color
                feature_to_cluster_name[feat] = cluster_name

        # Map colors for the bars and identify which clusters to put in the legend
        bar_colors = []
        for feat in data['feature']:
            color = feature_to_color.get(feat, default_color)
            bar_colors.append(color)
            
            # If this feature belongs to a known cluster, add it to our legend list
            c_name = feature_to_cluster_name.get(feat)
            if c_name and c_name not in present_clusters:
                present_clusters[c_name] = color

        # --- PLOTTING ---
        num_features = len(data)
        font_size = 11 if num_features < 40 else 10
        y_pos = np.arange(num_features)
        
        ax.barh(y_pos, data['mean'], color=bar_colors, align='center', alpha=0.9)
        ax.errorbar(data['mean'], y_pos, xerr=data['sem'], fmt='none', 
                    color='black', capsize=2, elinewidth=0.6)
        
        # --- FORMATTING ---
        ax.set_yticks(y_pos)
        clean_labels = [str(f).replace('feat_', '') for f in data['feature']]
        ax.set_yticklabels(clean_labels, fontsize=font_size)
        ax.set_title(f"{emotion} (n={num_features})", fontsize=12, fontweight='bold', pad=4)
        ax.axvline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.3)
        sns.despine(ax=ax)

        # --- ADDING THE LEGEND ---
        if present_clusters:
            # Sort legend items by cluster number (Cluster_1, Cluster_2...)
            sorted_clusters = sorted(present_clusters.items(), key=lambda x: int(x[0].split('_')[1]))
            
            legend_elements = [Patch(facecolor=color, label=name.replace('_', ' ')) 
                               for name, color in sorted_clusters]
            
            # Adjust loc/bbox_to_anchor if legend overlaps with bars
            ax.legend(handles=legend_elements, loc='lower right', 
                      fontsize=8, frameon=True, facecolor='white', framealpha=0.7)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout(h_pad=0.2, w_pad=1.5)
    return fig

# Run function
fig = plot_all_stable_emotions_compact_colorV2(en_stable_trimmed, output, n_rows=3, n_cols=5, figsize=(40,22))
plt.show()



fig.savefig(os.path.join(FIG_PATH, "EN_Significant_Features_ClustersColor.svg"), dpi=300, bbox_inches='tight')

In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

def plot_stable_emotions_colored(df, cluster_dict, target_emotion=None, n_rows=None, n_cols=None, figsize=None):
    """
    Plots stable features colored by cluster assignment. 
    Can plot a comprehensive grid or isolate a single target emotion.
    """
    # 1. Handle Structural Layout Mode
    if target_emotion is not None:
        all_emotions = [target_emotion]
        n_rows, n_cols = 1, 1
        if figsize is None:
            figsize = (10, 8)
    else:
        if n_rows is None or n_cols is None:
            raise ValueError("You must specify n_rows and n_cols when running grid mode.")
        emo_cols = [c for c in df.columns if '_Stable_Feature' in c]
        all_emotions = [c.replace('_Stable_Feature', '') for c in emo_cols]
        if figsize is None:
            figsize = (22, 40)

    # 2. Subplot Setup
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    if n_rows * n_cols == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    # Shared Color Settings
    palette = sns.color_palette("tab20", 20)
    default_color = '#4a82d6' 
    i = -1  # Safe tracking variable for layout cleanup

    for i, emotion in enumerate(all_emotions):
        if i >= len(axes):
            break
            
        ax = axes[i]
        cols = [f'{emotion}_Stable_Feature', f'{emotion}_Mean_Importance', f'{emotion}_SEM']
        existing_cols = [c for c in cols if c in df.columns]
        
        if len(existing_cols) < 3:
            ax.axis('off')
            continue
            
        data = df[existing_cols].dropna().copy()
        if data.empty:
            ax.axis('off')
            continue
            
        data.columns = ['feature', 'mean', 'sem']
        data['abs_mean'] = data['mean'].abs()
        data = data.sort_values('abs_mean', ascending=True)
        
        num_features = len(data)
        font_size = 12 if num_features < 40 else 11
        
        # Adjust panel vertical distribution dynamically for isolated large lists
        if target_emotion is not None and num_features > 30 and figsize == (10, 8):
            fig.set_figheight(num_features * 0.28)

        # 3. Cluster Coloring Logic
        emo_clusters = cluster_dict.get(emotion, {}).get('clusters', {})
        feature_to_color = {}
        feature_to_cluster_name = {}
        present_clusters = {} 

        for cluster_name, features in emo_clusters.items():
            try:
                cluster_num = int(cluster_name.split('_')[1])
                color = palette[(cluster_num - 1) % len(palette)]
            except (IndexError, ValueError):
                color = default_color
            
            for feat in features:
                feature_to_color[feat] = color
                feature_to_cluster_name[feat] = cluster_name

        bar_colors = []
        for feat in data['feature']:
            color = feature_to_color.get(feat, default_color)
            bar_colors.append(color)
            c_name = feature_to_cluster_name.get(feat)
            if c_name and c_name not in present_clusters:
                present_clusters[c_name] = color

        # 4. Rendering
        y_pos = np.arange(num_features)
        ax.barh(y_pos, data['mean'], color=bar_colors, align='center', alpha=0.9, edgecolor='none')
        ax.errorbar(data['mean'], y_pos, xerr=data['sem'], fmt='none', 
                    color='black', capsize=2, elinewidth=0.6)
        
        # Formatting
        ax.set_yticks(y_pos)
        clean_labels = [str(f).replace('feat_', '') for f in data['feature']]
        ax.set_yticklabels(clean_labels, fontsize=font_size)
        ax.set_title(f"{emotion} (n={num_features})", fontsize=14, fontweight='bold', pad=8)
        ax.set_xlabel('Mean Importance', fontsize=11, fontweight='bold')
        ax.axvline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.3)
        sns.despine(ax=ax)

        # 5. Panel-Specific Legend Positioning
        if present_clusters:
            sorted_clusters = sorted(present_clusters.items(), key=lambda x: int(x[0].split('_')[1]))
            legend_elements = [Patch(facecolor=color, label=name.replace('_', ' ')) 
                               for name, color in sorted_clusters]
            ax.legend(handles=legend_elements, loc='lower right', 
                      fontsize=9, frameon=True, facecolor='white', framealpha=0.8)

    # 6. Grid Cleanup Window
    if target_emotion is None:
        for j in range(i + 1, len(axes)):
            axes[j].set_visible(False)

    plt.tight_layout(h_pad=0.2, w_pad=1.5)
    return fig

In [ ]:
target_emo = "Joy"

# Generates clean standalone panels
fig = plot_stable_emotions_colored(en_stable_trimmed, output, target_emotion=target_emo)
plt.show()

# Saves safely as individual files
save_name = f"EN_Significant_Features_ClustersColor_{target_emo}.svg"
# fig.savefig(os.path.join(FIG_PATH, save_name), dpi=300, bbox_inches='tight')

# 26. Combine feature values and clusterings into 1 polar plot per emotion, with cluster-based coloring and a legend.

In [ ]:
from circular_emotion_wrapper_topfeat_quart import plot_one_emotion, plot_all_emotions


# ── Single emotion : Adoration ────────────────────────────────────────────────────────
# fig, ax = plot_one_emotion("Adoration", output, en_stable_trimmed,
#                            figsize=(10, 10),
#                            category_label_fontsize=14,  
#                            save_path=FIG_PATH + "circular_Adoration.svg")


# fig, ax = plot_one_emotion("AestheticAppreciation", output, en_stable_trimmed,
#                            figsize=(10, 10),
#                            category_label_fontsize=14,  
#                            save_path=FIG_PATH + "circular_Aesthetic.svg")

fig, ax = plot_one_emotion("Joy", output, en_stable_trimmed,
figsize=(10, 10),
category_label_fontsize=14,  
save_path=FIG_PATH + f"circular_Joy_{timestamp}.pdf",
)

# -------------for Emo 7 that Joy has low performance, we try Amusment --------------
fig, ax = plot_one_emotion("Amusement", output, en_stable_trimmed,
figsize=(10, 10),
category_label_fontsize=14,  
save_path=FIG_PATH + f"circular_Amusement_{timestamp}.pdf",
)

fig, ax = plot_one_emotion("Satisfaction", output, en_stable_trimmed,
figsize=(10, 10),
category_label_fontsize=14,  
save_path=FIG_PATH + f"circular_Satisfaction_{timestamp}.pdf",
)


plot_all_emotions(
    output, en_stable_trimmed,
    grid=True, n_cols=3,
    figsize=(18, 28),
    # ref_ring_values=[1, 2, 3, 4, 5],
    show_legend=False,                 # cleaner for grid view
    category_label_fontsize=14,        # bump labels even more if needed

    # save_grid_path=None,
    save_grid_path=FIG_PATH + f"circular_grid_all_blackring_{timestamp}.pdf",

)



## 27.Extract top emotions that are beyod first quartile 

In [ ]:
from circular_emotion_wrapper_topfeat_quart import (
    extract_top_features,
    extract_top_features_all_emotions,
)

# All emotions at once → first ring (25% of each emotion's max|importance|)
result = extract_top_features_all_emotions(output, en_stable_trimmed, ring_index=0)

# Combined long table — every feature, every emotion, above the cutoff
result["long_df"]


# Per-emotion dict for indexed access
result["by_emotion"]["Adoration"]

# Auto ring values per emotion (matches the dashed circles in the plot)
result["ring_values"]["Adoration"]   # array([1.20, 2.40, 3.60, 4.80])

# Counts summary
result["summary"]
#   emotion   n_features_total  n_above_threshold  threshold_used  ring_1  ring_2  ring_3  ring_4
#   Adoration 8                 6                  1.20            1.20    2.40    3.60    4.80
#   ...



## show the results in table format by extracting max  value within each cluster 

In [ ]:


# ── Strip Roman-numeral suffix so "Cortical Alpha I/II/III" all collapse ─────
ROMAN_SUFFIX = re.compile(r"\s+(?:I{1,3}|IV|V|VI{0,3}|IX|X)$")

agg = result["long_df"].copy()
agg["cluster_base"] = agg["cluster_name"].str.replace(ROMAN_SUFFIX, "", regex=True)

# ── Aggregate by emotion × cluster_base ──────────────────────────────────────
agg = (
    agg.groupby(["emotion", "cluster_base"], as_index=False)
       .agg(
           n_features=("feature", "size"),
           # Finds the original signed importance value that has the maximum absolute magnitude
           max_importance=("importance", lambda x: x.loc[x.abs().idxmax()]),
       )
)

agg["cell"] = agg.apply(
    lambda r: f"{r['max_importance']:+.2f} (n={r['n_features']})", axis=1
)

# ── Pivots (rows = cluster name, cols = emotion) ─────────────────────────────
pivot_num = agg.pivot_table(index="cluster_base", columns="emotion",
                            values="max_importance", aggfunc="first")
pivot_txt = agg.pivot_table(index="cluster_base", columns="emotion",
                            values="cell", aggfunc="first")

# Order rows by max |importance| across emotions (most impactful on top)
order = pivot_num.abs().max(axis=1).sort_values(ascending=False).index
pivot_num = pivot_num.loc[order]
pivot_txt = pivot_txt.loc[order]

vmax = float(pivot_num.abs().max().max())

# ── Styled display ───────────────────────────────────────────────────────────
styled = (
    pivot_txt.fillna("").style
        .background_gradient(cmap="RdBu_r", vmin=-vmax, vmax=vmax,
                             axis=None, gmap=pivot_num.values)
        .set_caption("Max absolute importance per cluster (rows) × emotion (columns) "
                     "— clusters merged by name, empty = below first ring")
        .set_properties(**{"font-size": "10pt", "text-align": "center",
                           "min-width": "85px", "border": "1px solid #ddd"})
        .set_table_styles([
            {"selector": "th.col_heading",
             "props": [("writing-mode", "vertical-rl"),
                       ("transform", "rotate(180deg)"),
                       ("padding", "4px"), ("font-size", "10pt")]},
            {"selector": "th.row_heading",
             "props": [("text-align", "left"), ("padding-right", "8px"),
                       ("font-size", "10pt"), ("white-space", "nowrap")]},
            {"selector": "caption",
             "props": [("caption-side", "top"),
                       ("font-weight", "bold"), ("padding", "6px")]},
        ])   
)       
display(styled)

In [ ]:


cmap = plt.get_cmap("RdBu_r")
norm = Normalize(vmin=-vmax, vmax=vmax)

n_rows, n_cols = pivot_txt.shape
fig, ax = plt.subplots(
    figsize=(0.95 * (n_cols + 2), 0.42 * (n_rows + 2)),
    dpi=200,
)
ax.set_axis_off()

# Build cell colors from the numeric pivot
cell_colors = [
    [cmap(norm(pivot_num.iat[r, c])) if pd.notna(pivot_num.iat[r, c])
     else (1, 1, 1, 1)
     for c in range(n_cols)]
    for r in range(n_rows)
]

# Cell text from the text pivot
cell_text = pivot_txt.fillna("").values.tolist()

table = ax.table(
    cellText=cell_text,
    cellColours=cell_colors,
    rowLabels=pivot_txt.index,
    colLabels=pivot_txt.columns,
    loc="center",
    cellLoc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1.0, 1.4)

# ── ADJUST TEXT PROPERTIES AND HEADERS ──────────────────────────────────────
for (r, c), cell in table.get_celld().items():
    if r == 0:  # Header row
        cell.set_text_props(rotation=45, ha="right", va="bottom")
        cell.set_height(0.13)
        
    elif r > 0 and c >= 0:  # Data cells
        val = pivot_num.iat[r - 1, c]  # r-1 accounts for the header row offset
        if pd.notna(val):
            # If the value's magnitude is > 50% of vmax, switch text color to white
            # Tune the 0.50 threshold lower (e.g., 0.45) if you want more cells to have white text
            if abs(val) / vmax > 0.50:
                cell.set_text_props(color="white")

ax.set_title("Max absolute importance per cluster × emotion", pad=20, fontweight="bold")

fig.savefig(FIG_PATH + "emotion_cluster_pivot_max.svg", bbox_inches="tight")
fig.savefig(FIG_PATH + "emotion_cluster_pivot_max.png", bbox_inches="tight", dpi=200)
plt.show()

## Feature rank correlation across emotions

In [ ]:
from scipy.cluster.hierarchy import fcluster

def plot_feature_rank_correlation(stable_df, feature_subset=None,
                                  exclude_emotions=None, figsize=(12, 12)):
    """
    Compute and visualise Spearman rank correlation of feature ordering
    across emotions.  Optionally restrict to *feature_subset* and/or
    exclude certain emotions.

    Returns (corr_matrix, rank_df, cluster_df)
    where cluster_df has columns: emotion, cluster_L2, cluster_L3, cluster_L4, cluster_L5
    """
    feat_cols = [c for c in stable_df.columns if '_Stable_Feature' in c]
    emotions  = [c.replace('_Stable_Feature', '') for c in feat_cols]
    emotions  = list(dict.fromkeys(emotions))

    if exclude_emotions:
        if isinstance(exclude_emotions, str):
            exclude_emotions = [exclude_emotions]
        emotions = [e for e in emotions if e not in exclude_emotions]

    # Build rank matrix
    rank_data = {}
    for emo in emotions:
        ordered  = stable_df[f'{emo}_Stable_Feature'].dropna().tolist()
        rank_map = {f: i + 1 for i, f in enumerate(ordered)}
        target_feats = feature_subset if feature_subset is not None else ordered
        rank_data[emo] = {f: rank_map.get(f, np.nan) for f in target_feats}

    rank_df = pd.DataFrame(rank_data)
    if rank_df.isna().any().any():
        rank_df = rank_df.fillna(rank_df.max().max() + 1)

    corr = rank_df.corr(method='spearman')
    

    
    n_feats = len(feature_subset) if feature_subset else len(rank_df)
    excl    = f" (excl. {', '.join(exclude_emotions)})" if exclude_emotions else ''
    title   = f"Clustered Rank Correlation — {n_feats} features{excl}"

    
    g = sns.clustermap(
        corr, method='ward', metric='euclidean',
        figsize=figsize, annot=True, fmt='.2f',
        cmap='Spectral_r',
        linewidths=.5,
        dendrogram_ratio=0.15,
        cbar_pos=(0.02, 0.85, 0.03, 0.15)
    )
    plt.setp(g.ax_heatmap.get_xticklabels(), fontsize=14,rotation=45, ha='right')
    plt.setp(g.ax_heatmap.get_yticklabels(), fontsize=14,rotation=0)
    g.fig.suptitle(title, fontsize=16, y=1.02)
    plt.show()

    # ── Cluster assignments at 4 depth levels ───────────────────────────────
    linkage = g.dendrogram_row.linkage

    cluster_df = pd.DataFrame({'emotion': corr.index.tolist()})
    for depth in range(2, 6):   # levels 2, 3, 4, 5
        cluster_df[f'cluster_L{depth}'] = fcluster(
            linkage, t=depth, criterion='maxclust'
        )

    cluster_df = cluster_df.sort_values('cluster_L2').reset_index(drop=True)

    print("\nEmotion clusters across 4 depth levels:")
    for depth in range(2, 6):
        col = f'cluster_L{depth}'
        print(f"\n  Level {depth} ({depth} clusters):")
        for cid, grp in cluster_df.groupby(col):
            print(f"    Cluster {cid}: {grp['emotion'].tolist()}")

    return corr, rank_df, cluster_df, g

In [ ]:
corr, rank_df, cluster_df, fig= plot_feature_rank_correlation(en_stable_trimmed)

plt.show()
fig.savefig(os.path.join(FIG_PATH, f"Rank_Correlation_FeaturesTrimmed_AcrossEmotions_{PID}_{timestamp}_V2.svg"), dpi=300, bbox_inches='tight')


common_features_trimmed, specific_stable_df, distinct_df = get_feature_sets(en_stable_trimmed)
print(common_features_trimmed)


# # Restrict to common features, excluding one emotion
order_corr_common, rank_df_common, cluster_df_common, fig_common = plot_feature_rank_correlation(
    en_stable_trimmed, feature_subset=common_features_trimmed, exclude_emotions=[]
)

plt.show()
fig_common.savefig(os.path.join(FIG_PATH, f"Rank_Correlation_CommonFeaturesTrimmed_AcrossEmotions_{PID}_{timestamp}.svg"), dpi=300, bbox_inches='tight')

##  Compute relationships between clustering results of physio features vs. behavioral ratings

In [ ]:
target_raw = mat['agg_data']['sr'][0, 0]
target_dropped = np.delete(target_raw, nan_idx_task, axis=0).astype(int)

# Emotion labels
emotions     = mat['agg_data']['conditions'][0, 0]
emos         = [item[0] for item in emotions[0]]
select_emos  = [emos[i] for i in EMOTION_COLS]
sel_target   = target_dropped[:, EMOTION_COLS]

# Feature column names (exclude meta columns)
features_names = [col for col in all_feat_norm.columns if col not in META_COLS]
print(features_names)


if PID == 'emo07':
    sel_target = np.delete(sel_target, 6, axis=1)  # drop the 7th emotion (index 7)
    select_emos = [e for i, e in enumerate(select_emos) if i != 6]  # also drop label


print("Emotions:", select_emos)
print(f"Target shape: {sel_target.shape}  |  features: {len(features_names)}")


binary_target = (sel_target != 0).astype(int)
binary_target.shape

# compute and plot the clustering for the target ratings 

In [ ]:


def cluster_numpy_correlations(data_array, col_names=None, figsize=(12, 12)):
    """
    Computes Pearson correlation between columns of a numpy array,
    plots a clustermap, and returns cluster assignments.
    """
    # 1. Create DataFrame for easier handling of column names
    if col_names is None:
        col_names = [f"Col_{i}" for i in range(data_array.shape[1])]
    
    df = pd.DataFrame(data_array, columns=col_names)

    # 2. Compute Correlation Matrix (Pearson)
    # This correlates columns based on the values across rows
    corr_matrix = df.corr()

    # 3. Plot Clustermap
    # 'ward' method and 'euclidean' metric are standard for hierarchical clustering
    g = sns.clustermap(
    corr_matrix, 
    method='ward', 
    metric='euclidean',
    figsize=figsize,      # Increased size for clarity
    annot=True, 
    fmt='.2f', 
    cmap='Spectral_r',
    linewidths=0.5,        # Adds thin lines between cells to prevent "bleeding"
    dendrogram_ratio=0.15, # Reduces the size of the tree lines
    cbar_pos=(0.02, 0.85, 0.03, 0.15) # [left, bottom, width, height] - moves legend
    )

    # Tilt x-axis and fix alignment
    plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha='right', fontsize=14)
    plt.setp(g.ax_heatmap.get_yticklabels(), rotation=0, fontsize=14)

    # Add a title with extra padding
    g.fig.suptitle("Covariance Matrix of the self-reports", fontsize=16, y=1.05)

    # 4. Extract Cluster Assignments
    # We use the linkage matrix generated by the clustermap
    linkage = g.dendrogram_row.linkage
    
    cluster_results = pd.DataFrame({'column_name': corr_matrix.index})
    
    # Extract clusters for different depths (2 through 5)
    for depth in range(2, 6):
        cluster_results[f'cluster_L{depth}'] = fcluster(
            linkage, t=depth, criterion='maxclust'
        )

    return corr_matrix, cluster_results,g

# -

In [ ]:

corr, clusters,fig = cluster_numpy_correlations(sel_target, col_names=select_emos, figsize=(12, 12))

fig.savefig(os.path.join(FIG_PATH, f"Covariance of Self-Reports_{PID}_{timestamp}.svg"), dpi=300, bbox_inches='tight')


corr, clusters,fig = cluster_numpy_correlations(binary_target, col_names=select_emos, figsize=(12, 12))
fig.savefig(os.path.join(FIG_PATH, f"Covariance of Binary Self-Reports_{PID}_{timestamp}.svg"), dpi=300, bbox_inches='tight')


print(clusters)

## plot consensus between neurophysiology and self-reports

In [ ]:

def plot_multimodal_consensus_network(df_neuro, df_behavior, level='cluster_L5', save_path=None):
    # 1. Align column names and define data sources
    # df1 = Neurophysiology, df2 = Behavior
    d_neuro = df_neuro.rename(columns={'emotion': 'name', 'column_name': 'name'})
    d_behav = df_behavior.rename(columns={'emotion': 'name', 'column_name': 'name'})
    
    G = nx.Graph()
    emotions = d_neuro['name'].tolist()
    G.add_nodes_from(emotions)
    
    # 2. Build edges based on specific comembership
    for i in range(len(emotions)):
        for j in range(i + 1, len(emotions)):
            emo1, emo2 = emotions[i], emotions[j]
            
            # Shared in Neurophysiology (df1)
            in_neuro = d_neuro.loc[d_neuro['name'] == emo1, level].values[0] == \
                       d_neuro.loc[d_neuro['name'] == emo2, level].values[0]
            
            # Shared in Behavior (df2)
            in_behav = d_behav.loc[d_behav['name'] == emo1, level].values[0] == \
                       d_behav.loc[d_behav['name'] == emo2, level].values[0]
            
            if in_neuro and in_behav:
                # Both models agree
                G.add_edge(emo1, emo2, weight=5, color='seagreen', type='consensus')
            elif in_neuro:
                # Only Neurophysiology clusters them
                G.add_edge(emo1, emo2, weight=1.2, color='royalblue', type='neuro')
            elif in_behav:
                # Only Behavior clusters them
                G.add_edge(emo1, emo2, weight=1.2, color='crimson', type='behav')

    # 3. Component-Wise Manual Placement
    components = list(nx.connected_components(G))
    pos = {}
    # Centers spaced out to "distance the middle"
    centers = [(-0.8, 0), (0.8, 0), (0, 0.9), (0, -0.9)] 
    
    for i, nodes in enumerate(components):
        subgraph = G.subgraph(nodes)
        # Spread nodes within the island
        local_pos = nx.spring_layout(subgraph, k=0.25, iterations=150, seed=42)
        
        center = centers[i % len(centers)]
        for node in nodes:
            pos[node] = local_pos[node] + np.array(center)

    # 4. Visualization
    plt.figure(figsize=(14, 11))
    
    edges = G.edges(data=True)
    draw_weights = [6 if e[2]['type'] == 'consensus' else 3.5 for e in edges]

    colors = [e[2]['color'] for e in edges]


# Then apply them in the draw function

    # Draw edges
    nx.draw_networkx_edges(G, pos, width=draw_weights, edge_color=colors, alpha=0.75)
    
    # Draw nodes
    nx.draw_networkx_nodes(G, pos, node_size=800, node_color='white', edgecolors='black', linewidths=1.5)
    
    # Draw labels with background halo
    nx.draw_networkx_labels(G, pos, font_size=12, font_weight='bold', 
                            bbox=dict(facecolor='white', edgecolor='none', alpha=0.85, pad=0.3))

    # 5. Legend with new semantic labels
    legend_elements = [
        Line2D([0], [0], color='seagreen', lw=4, label='Consensus (Both Models)'),
        Line2D([0], [0], color='royalblue', lw=2.5, label='Neurophysiology Only'),
        Line2D([0], [0], color='crimson', lw=2.5, label='Self-Report Only')
    ]
    plt.legend(handles=legend_elements, loc='upper right', frameon=True, fontsize=11, title="Clustering Logic")

    plt.title(f"Multimodal Clustering Consensus ({level})", fontsize=18, pad=30)
    plt.axis('off')
    
    # Dynamic Zoom
    all_coords = np.array(list(pos.values()))
    plt.xlim(all_coords[:, 0].min() - 0.25, all_coords[:, 0].max() + 0.25)
    plt.ylim(all_coords[:, 1].min() - 0.25, all_coords[:, 1].max() + 0.25)
    
    plt.tight_layout()

    if save_path:
        # facecolor='white' ensures the background isn't transparent in PNGs
        # final_filename = f"{save_path}consensus_networkx_{level}.svg"
        
        # Save the figure
        plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"Figure successfully saved to: {save_path}")

    plt.show()   



In [ ]:
# plot_multimodal_consensus_network(cluster_df, clusters, level='cluster_L4', save_path=FIG_PATH+f"Consensus_Network_{PID}_{timestamp}.svg")÷
plot_multimodal_consensus_network(cluster_df, clusters, level='cluster_L4')

## Consensus with another version of visualization

## 37. plot the clustered feature for clustered emotion based on majority voting (Borda plot), colors are consistent with feature clusters 

In [ ]:
import re

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Pull the cluster colour scheme straight from the circular-ring module so the
# two visualisations stay perfectly in sync. Adjust this import path to wherever
# CATEGORY_COLORS / categorize_feature actually live in your project (in your
# pipeline this is the module imported as `gm_circular_emotion_cluster_ring_black`).
# from gm_circular_emotion_cluster_ring_black import (
#     CATEGORY_COLORS,
#     CATEGORY_ORDER,
#     categorize_feature,
#     _dominant_region,
#     _shade_for_region,
# )

from circular_emotion_cluster_ring_black_colors import (
    CATEGORY_COLORS,
    CATEGORY_ORDER,
    categorize_feature,
    _dominant_region,
    _shade_for_region,
)


# Extra recognition fallbacks. These ONLY fire when gm's own categoriser /
# region detector can't resolve a single feature name (i.e. it would otherwise
# fall back to gray / an unshaded base). The COLOURS themselves are still taken
# straight from gm's CATEGORY_COLORS / _shade_for_region, so the palette matches
# gm_circular_ring_black exactly — this just makes sure cardiovascular, blood
# pressure, and posterior-vs-frontal EEG features actually reach those colours.
# Cardiovascular detection. The headline tokens are matched as whole words
# (split on non-alphanumeric), so a bare "HR", "HRV", "ECG" hits — as does
# "HR_mean", "mean_HR", "ECG_rate" — while words like "threshold" do NOT.
_HR_TOKENS = {"hr", "hrv", "ecg", "ibi", "bpm", "rri", "nni"}
# Longer, unambiguous substrings can still match anywhere in the name.
_HR_SUBSTR = ("heart", "cardio", "vascular", "pulse",
              "rmssd", "sdnn", "pnn", "tachogram")
_BP_HINTS = ("_bp", "bp_", "sbp", "dbp", "systol", "diastol",
             "blood_pressure", "pressure")

# Local colour overrides on top of gm's CATEGORY_COLORS. Heart Rate is forced
# to a *very* dark green here (gm's default #0F6E56 read too light/gray).
_COLOR_OVERRIDES = {
    "Heart Rate": "#0A3B2A",   # very dark green
}

_FRONTAL_HINTS   = ("_front", "front", "anterior", "prefront")
_POSTERIOR_HINTS = ("_post", "post", "occip", "pariet")
_MID_HINTS       = ("_mid", "mid", "central", "vertex")


def _resolve_category(feature_name):
    """gm's categoriser first; only add fallbacks when it returns 'Other'.

    Also corrects low-alpha that gm's literal 'alpha_low' check misses: a name
    like 'low_alpha_...' or 'alpha-low' otherwise falls through to plain
    'EEG Alpha' (red) and collapses onto regular alpha. We push it back to
    'EEG Alpha-Low' so it keeps gm's distinct magenta base (#C24FA0).
    """
    cat = categorize_feature(feature_name)
    n = feature_name.lower()
    if cat == "EEG Alpha":
        tokens = set(re.split(r"[^a-z0-9]+", n))
        if ("low" in tokens or "lowalpha" in n or "alphalow" in n
                or "alpha_low" in n or "low_alpha" in n):
            return "EEG Alpha-Low"
        return cat
    if cat != "Other":
        return cat
    tokens = set(re.split(r"[^a-z0-9]+", n))
    if tokens & _HR_TOKENS or any(h in n for h in _HR_SUBSTR):
        return "Heart Rate"
    if any(h in n for h in _BP_HINTS):
        return "BP"
    return cat


def _resolve_region(feature_name):
    """gm's region detector first; only add fallbacks when it returns None."""
    region = _dominant_region([feature_name])
    if region is not None:
        return region
    n = feature_name.lower()
    if any(h in n for h in _FRONTAL_HINTS):
        return "Frontal"
    if any(h in n for h in _POSTERIOR_HINTS):
        return "Posterior"
    if any(h in n for h in _MID_HINTS):
        return "Mid"
    return None


def _feature_color(feature_name):
    """Return the gm_circular_ring_black colour for a single feature.

    Non-EEG features use their flat CATEGORY_COLORS value (e.g. BP -> brown
    #7A2E20, Heart Rate -> dark green #0F6E56); EEG features get the same
    region-based shading used in the circular-ring plots — frontal lightened,
    posterior darkened — so posterior alpha reads dark red and frontal alpha
    reads a lighter red, identical to gm.
    """
    cat = _resolve_category(feature_name)
    base = _COLOR_OVERRIDES.get(cat, CATEGORY_COLORS.get(cat, "#888888"))
    if cat.startswith("EEG "):
        region = _resolve_region(feature_name)
        base = _shade_for_region(base, region)
    return base


def _wrap_label(feature):
    """Break a multi-word cluster name across lines, one word per line.

    A two-word label like "Frontal Theta" becomes "Frontal\\nTheta" (second
    word on a second line); single-word labels are left untouched.
    """
    return "\n".join(str(feature).split())


def plot_network_clusters_radial(df_weights, cluster_dict, save_path=None,
                                    show_category_legend=True):
    """
    Applies rank-order aggregation (Borda Count) to raw feature weights
    and generates aligned Nightingale Rose charts for each emotion cluster.

    Bars are now coloured by physiological category using the same scheme as
    gm_circular_ring_black (CATEGORY_COLORS + EEG region shading). Bar *height*
    still encodes the normalised consensus score, while bar *colour* encodes the
    feature category.
    """
    # Absolute weights to Ranks
    df_abs = df_weights.abs()
    ranks = df_abs.rank(ascending=False)

    # Ranks to Consensus Points
    max_features = len(df_weights)
    points_df = max_features - ranks + 1
    points_df = points_df.fillna(0)

    # Establish a GLOBAL feature order
    global_scores = points_df.sum(axis=1).sort_values(ascending=False)
    global_features = global_scores[global_scores > 0].index.tolist()
    num_vars = len(global_features)

    # ── Category colours (matched to gm_circular_ring_black) ────────────────
    # One colour per global feature, computed once since the feature order is
    # shared across every subplot.
    feature_colors = [_feature_color(feat) for feat in global_features]

    angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
    width = 2 * np.pi / num_vars

    cols = min(3, len(cluster_dict))
    rows = int(np.ceil(len(cluster_dict) / cols))

    fig, axes = plt.subplots(rows, cols, subplot_kw={'projection': 'polar'},
                             figsize=(10 * cols, 10 * rows))
    axes = np.array(axes).flatten()

    # ── Tuned positions ─────────────────────────────────────────────────────
    LABEL_R   = 108     # labels just outside the data ring (back to original)
    AXIS_RMAX = 130     # only slightly past LABEL_R — kills the empty band
    TITLE_Y   = 1.10    # title close to the chart, but above the topmost label

    for ax, (cluster_name, emotions) in zip(axes, cluster_dict.items()):
        valid_emotions = [e for e in emotions if e in points_df.columns]
        if not valid_emotions:
            ax.set_title(f"{cluster_name}\n(No Data)", y=TITLE_Y,
                         fontweight='bold', fontsize=20)
            ax.set_yticklabels([])
            ax.set_xticklabels([])
            continue

        cluster_score = points_df[valid_emotions].sum(axis=1)
        max_possible = max_features * len(valid_emotions)
        normalized_score = (cluster_score / max_possible) * 100

        scores = [normalized_score.get(feat, 0) for feat in global_features]
        scores_array = np.array(scores)

        # Colour by physiological category (matches gm_circular_ring_black)
        # instead of the previous plasma_r score gradient.
        ax.bar(angles, scores_array, width=width, bottom=0.0, alpha=0.85,
               edgecolor='white', color=feature_colors)

        ax.set_theta_offset(np.pi / 2)
        ax.set_theta_direction(-1)

        # Expand axis slightly past the labels so nothing gets clipped,
        # but not so far that we get a big empty band like before.
        ax.set_ylim(0, AXIS_RMAX)
        ax.set_yticks([25, 50, 75, 100])
        ax.set_yticklabels(['25', '50', '75', '100'], color='grey', size=9)

        # Hide the outer polar spine (it would sit at AXIS_RMAX); draw a
        # subtle ring at 100 to mark the actual data boundary instead.
        ax.spines['polar'].set_visible(False)
        ax.plot(np.linspace(0, 2*np.pi, 200), [100]*200,
                color='#bbbbbb', linewidth=0.9, zorder=1)

        ax.set_title(f"{cluster_name}", y=TITLE_Y,
                     fontweight='bold', fontsize=20)

        ax.set_xticks(angles)
        ax.set_xticklabels([])

        # Feature labels — bigger font, closer to outer ring
        for angle, feature in zip(angles, global_features):
            theta_visual_rad = np.pi/2 - angle
            theta_visual_deg = np.degrees(theta_visual_rad) % 360

            if 90 < theta_visual_deg < 270:
                ha = 'right'
                rot = theta_visual_deg + 180
            else:
                ha = 'left'
                rot = theta_visual_deg

            # Two-word (multi-word) cluster names break onto a second line.
            wrapped_feature = _wrap_label(feature)

            ax.text(angle, LABEL_R, wrapped_feature, ha=ha, va='center',
                    rotation=rot, rotation_mode='anchor',
                    fontsize=18)             # ← bumped from 13 → 15

    for i in range(len(cluster_dict), len(axes)):
        fig.delaxes(axes[i])

    # ── Category legend so the new colours are decodable ────────────────────
    # IMPORTANT: build the legend from the SAME resolver the bars use
    # (_resolve_category) and the SAME colour path (_COLOR_OVERRIDES first),
    # otherwise a bar coloured via a fallback (e.g. a "Posterior Beta" cluster
    # name that gm's raw categorize_feature can't place) would appear on the
    # chart with no matching legend entry. (Drop show_category_legend=False
    # if you don't want this.)
    if show_category_legend:
        present = {_resolve_category(f) for f in global_features}
        ordered_cats = [c for c in CATEGORY_ORDER if c in present]
        ordered_cats += [c for c in present if c not in ordered_cats]
        handles = [
            mpatches.Patch(
                facecolor=_COLOR_OVERRIDES.get(c, CATEGORY_COLORS.get(c, "#888888")),
                edgecolor="white", label=c)
            for c in ordered_cats
        ]
        fig.legend(handles=handles, loc="lower center",
                   ncol=min(len(handles), 6), fontsize=18,
                   bbox_to_anchor=(0.5, -0.01), framealpha=0.95,
                   edgecolor="#cccccc")

    plt.tight_layout(pad=2.5, h_pad=3.5, w_pad=2.5)

    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"Figure successfully saved to: {save_path}")

    plt.show()

In [ ]:

#  4. EXECUTE & SAVE
# ==========================================
# Change the extension to .pdf if you prefer a vector graphics format for your paper
file_name = f"emotion_cluster_rank_networkx_L4_featureprofiles_{PID}_{timestamp}.svg" 


#-------------------------Emo 5--------------------------

if PID == 'emo05':
    plot_network_clusters_radial(pivot_num, {
        'Entranced & Aesthetic': [ 'Entrancement', 'AestheticAppreciation'],
        'Amus-Int-Admi_Awe_Re_Sa/Group2' : ['Amusement','Interest', 'Admiration', 'Awe',  'Relief','Satisfaction'],
        'Disengaged': ['Confusion', 'Boredom'],
        'Joy': ['Joy'],
        'Adoration' : ['Adoration'], 
        'Excitement' : ['Excitement'],
        'Awekwardness': ['Awkwardness'],
        'calmness': ['Calmness']},
        save_path=FIG_PATH + file_name)


# -------------------------Emo 6-------------------------- based on jaccard on trimmed data 
# plot_network_clusters_radial(pivot_num, {
#     ' Interest and Aesthetic/ Group 1': ['Amusement', 'Entrancement', 'AestheticAppreciation', 'Interest', 'Satisfaction'],
#     'Disengaged/Group 2': ['Confusion', 'Awkwardness'],
#     'Group 3': ['Admiration', 'Calmness'],
#     'High Arousal/Group 4': ['Excitement'],
#     'Group 5' : ['Joy','Adoration','Relief'], 
#     'Boredom/Group 6' : ['Boredom'],
#     'Excitement/Group 7': ['Excitement'], 
#     'Awe/Group 8': ['Awe']
#     },
#     save_path=FIG_PATH + file_name)    


# -------------------------Emo 6-------------------------- based on rank correlaiton on trimmed data 
elif PID == 'emo06':
    plot_network_clusters_radial(pivot_num, {
        'Interest and Aesthetic/ Group 1': ['Amusement', 'Entrancement', 'AestheticAppreciation', 'Interest', 'Satisfaction'],
        'Disengaged (Confusion-Awkwardness-Boredom)/Group 2': ['Boredom','Confusion', 'Awkwardness'],
        'Relief-Calmness/Group 3': ['Relief','Calmness'],
        'Excitement/Group 4': ['Excitement'],
        'Joy-Adoration-Admiration/Group 5' : ['Joy','Adoration','Admiration'], 
        'Awe/Group 6': ['Awe']
        },
        save_path=FIG_PATH + file_name)    

# -------------------------Emo 7-------------------------- based on rank correlaiton on trimmed data 
# elif PID == 'emo07':
#     plot_network_clusters_radial(pivot_num, {
#         'Excitement-Aesthetic/Group 1': ['Excitement','AestheticAppreciation','Admiration'],
#         'Calmness-Joy /Group 2' : ['Joy','Calmness'],
#         'Interest-Awe-Entrancement/ Group 3': ['Awe', 'Entrancement', 'Interest'],
#         'Amusement-Awkwardness/Group 4': ['Amusement', 'Awkwardness'],
#         # 'Admiration/Group 5': ['Admiration'],
#         'Joy /Group 5': ['Boredom'],
#         'Satisfaction/Group 6': ['Satisfaction'],
#         'Adoration/Group 7': ['Adoration'], 
#         'Confusion/Group 8': ['Confusion']
#         },
#         save_path=FIG_PATH + file_name)      

# ----------------------Emo 7 ------------------------ based on 6/22/2026 and fixing the folds for low sampling number 
elif PID == 'emo07':
    plot_network_clusters_radial(pivot_num, {
        'Excitement-Satisfaction/Group 1': ['Excitement','Satisfaction','Admiration'],
        'Calmness-Joy/Group 2' : ['Joy','Calmness'],
        'Interest-Awe-Entrancement/ Group 3': ['Awe', 'Entrancement', 'Interest'],
        'Amusement/Group 4': ['Amusement'],
        'Awkwardness/Group 5': ['Awkwardness'],
        'Boredom/Group 5': ['Boredom'],
        'Aesthetic/Group 6': ['AestheticAppreciation'],
        'Adoration/Group 7': ['Adoration'], 
        'Confusion/Group 8': ['Confusion']
        },
        save_path=FIG_PATH + file_name)         


    
# -------------------------Emo 1-------------------------- based on rank correlaiton on trimmed data 
elif PID == 'emo01':
    plot_network_clusters_radial(pivot_num, {
        'Excitement-Relief/Group 1': ['Excitement','Relief'],
        'Calmness-Entrancement-Aesthetic/Group 2' : ['Entrancement','Calmness','AestheticAppreciation'],
        'Interest and Awe/ Group 3': ['Awe', 'Satisfaction', 'Interest'],
        'engaged/Group 4': ['Amusement',  'Adoration'],
        'Admiration/Group 5': ['Admiration'],
        'Boredom /Group 6': ['Boredom'],
        'Awekwardness-Joy-Confusion/Group 7': ['Awkwardness', 'Joy','Confusion']
        },
    )
        # save_path=FIG_PATH + file_name)        

# 38. Save output of emotion clustering 

In [ ]:

RANKCORR_STORE_PATH = os.path.join(DATA_ROOT, "feature_rank_corr_store.pkl")

if os.path.exists(RANKCORR_STORE_PATH):
    with open(RANKCORR_STORE_PATH, "rb") as f:
        rank_corr_store = pickle.load(f)
else:
    rank_corr_store = {}

rank_corr_store[PID] = {
    "corr":       corr.copy(),
    "rank_df":    rank_df.copy(),
    "cluster_df": cluster_df.copy(),
    "pivot_num":  pivot_num.copy(),
}

with open(RANKCORR_STORE_PATH, "wb") as f:
    pickle.dump(rank_corr_store, f)

print(f"Saved rank-correlation outputs for {PID} → {RANKCORR_STORE_PATH}")
print(f"PIDs in store: {list(rank_corr_store)}")
print(f"Keys for {PID}: {list(rank_corr_store[PID])}")

# Load the feature ranks: 

In [ ]:
import os
import pickle

RANKCORR_STORE_PATH = os.path.join(DATA_ROOT, "feature_rank_corr_store.pkl")
with open(RANKCORR_STORE_PATH, "rb") as f:
    rank_corr_store = pickle.load(f)

corr       = rank_corr_store[PID]["corr"]
rank_df    = rank_corr_store[PID]["rank_df"]
cluster_df = rank_corr_store[PID]["cluster_df"]
pivot_num  = rank_corr_store[PID]["pivot_num"]   # feeds the radial plot

In [ ]:
pivot_num.head(20)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

def imagesc_targets(sel_target, select_emos, pid=None, save_path=None):
    """imagesc-style heatmap of samples (trials) x emotions for the rating targets.

    Yellow = low values, purple = high values, via the reversed viridis colormap.
    """
    n_trials, n_emos = sel_target.shape

    fig, ax = plt.subplots(figsize=(max(8, n_emos * 0.6), 10))

    im = ax.imshow(
        sel_target,
        aspect='auto',
        # cmap='viridis_r',          # yellow (low) -> purple (high)
        cmap='Spectral_r',          # 
        interpolation='nearest',
    )

    # x-axis: emotions
    ax.set_xticks(np.arange(n_emos))
    ax.set_xticklabels(select_emos, rotation=45, ha='right', fontsize=9)

    # y-axis: trials / samples
    ax.set_ylabel('Trials', fontsize=12)
    ax.set_xlabel('Emotions', fontsize=12)
    ax.set_title(f'Ratings — samples x emotions{f"  ({pid})" if pid else ""}',
                 fontsize=14)

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Rating', fontsize=11)

    fig.tight_layout()

    if save_path:
        timestamp = datetime.now().strftime("%Y%m%d")
        fname = f"{save_path}imagesc_targets_{pid or 'all'}_{timestamp}.svg"
        fig.savefig(fname, dpi=300, bbox_inches='tight')
        print(f"Saved -> {fname}")

    plt.show()
    return fig, ax


# sel_target and select_emos already exist (e.g. from cell 13 / cell 150)
imagesc_targets(sel_target, select_emos, pid=PID, save_path=FIG_PATH)

## Train null distribitions 

## Parallel computing on CPU

In [ ]:
from collections import Counter
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.exceptions import ConvergenceWarning
from sklearn.exceptions import UndefinedMetricWarning

from joblib import Parallel, delayed
import warnings
import logging
import time

warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', category=UndefinedMetricWarning)

# =====================================================================
# PERMUTATION TRAINING LOOP (100 Iterations) — PARALLEL (CPU) VERSION
# =====================================================================
#
# Parallelization strategy
# ------------------------
# Instead of looping serially over 100 permutations x n_folds, we flatten
# every (perm_iter, fold) pair into an independent task and hand the whole
# batch to joblib.Parallel. Each task shuffles its own training labels,
# runs your existing grid_search/train_models/evaluate_models unchanged,
# and returns its result. This is finer-grained than parallelizing only
# over the 100 permutations, which matters if folds differ in size/cost.
#
# Important: avoid NESTED parallelism. If grid_search/train_models
# internally use sklearn's n_jobs (e.g. GridSearchCV(n_jobs=...)), set
# those to 1 so you don't oversubscribe cores (outer joblib workers x
# inner sklearn workers = contention, not speed). This file assumes
# your helper functions already use n_jobs=1 internally (train_models is
# already called with n_jobs=1 below) — if grid_search takes an n_jobs
# argument too, make sure it's 1 as well.
#
# Reproducibility: np.random.permutation relies on global numpy random
# state, which is NOT safe to share across parallel worker processes
# (workers fork/spawn independently and may repeat sequences). Each task
# below gets its own numpy Generator seeded deterministically from
# (RANDOM_SEED, emotion index, perm_iter, fold), so results are
# reproducible AND independent across workers.

N_PERMUTATIONS = 100
N_JOBS = -1  # use all available cores; set to an integer to cap it

all_emo_perm_perf   = []   # Will store a list of 100 iterations per emotion
all_models_perm     = []   # Will store a list of 100 iterations per emotion


def _run_one_task(i_e, perm_iter, fold, emotion, X_tr, Y_tr, X_te, Y_te,
                   MODELS, PARAM_GRIDS, RANDOM_SEED, features_names):
    """
    Runs a single (permutation, fold) unit of work.
    Returns None if the fold is skipped (test set has <2 classes),
    otherwise returns (perm_iter, perf, trained).
    """
    if len(set(Y_te)) < 2:
        return None

    # Deterministic-but-independent RNG per task, so parallel workers
    # never repeat/collide on the same shuffle.
    seed = (RANDOM_SEED, i_e, perm_iter, fold)
    rng = np.random.default_rng(seed)
    Y_tr_shuffled = rng.permutation(Y_tr)

    gs_results = grid_search(
        emotion, MODELS, PARAM_GRIDS, X_tr, Y_tr_shuffled,
        random_seed=RANDOM_SEED, feature_list=features_names
    )
    gs_df = pd.DataFrame(gs_results)

    trained = train_models(
        gs_df, X_tr, Y_tr_shuffled,
        feature_names=features_names, n_jobs=1
    )

    perf = evaluate_models(trained, X_te, Y_te, X_train=X_tr)

    if isinstance(perf, list):
        for p in perf:
            if isinstance(p, dict):
                p['permutation_iter'] = perm_iter
    elif isinstance(perf, dict):
        perf['permutation_iter'] = perm_iter

    return perm_iter, perf, trained


for i_e, emotion in enumerate(select_emos):
    logging.info("==================================================")
    logging.info(f"Emotion {i_e+1}/{len(select_emos)}: {emotion}")
    logging.info("==================================================")

    # Build combined feature matrix: positive-labelled task rows + all baseline rows
    target_col = sel_target[:, i_e]
    pos_idx    = np.where(target_col == 1)[0]

    targets  = np.concatenate([
        target_col[pos_idx],
        np.zeros(len(base_feat_norm), dtype=int)
    ])
    features = pd.concat(
        [all_feat_norm.iloc[pos_idx], base_feat_norm], ignore_index=True
    )

    # Get folds normally (we preserve the exact same splits for all 100 permutations)
    x_train, y_train, x_test, y_test = get_folds_data_fixed_train(
        n_split=N_FOLDS, feat_pd=features, target=targets.astype(int),
        emotion_name=emotion
    )

    if x_train is None:
        # Fails feasibility check (M < 2)
        all_emo_perm_perf.append([])
        all_models_perm.append([])
        continue

    # ── BUILD FLAT TASK LIST: every (perm_iter, fold) pair ──
    tasks = []
    for perm_iter in range(N_PERMUTATIONS):
        for fold, (X_tr, Y_tr, X_te, Y_te) in enumerate(
            zip(x_train, y_train, x_test, y_test)
        ):
            tasks.append((perm_iter, fold, X_tr, Y_tr, X_te, Y_te))

    logging.info(f"  Dispatching {len(tasks)} (permutation x fold) tasks "
                 f"across n_jobs={N_JOBS}...")
    t0 = time.time()

    # ── RUN ALL TASKS IN PARALLEL ──
    # backend="loky" (the default) uses separate processes, which is the
    # right choice for CPU-bound sklearn work (sidesteps the GIL cleanly,
    # unlike threads).
    results = Parallel(n_jobs=N_JOBS, backend="loky", verbose=5)(
        delayed(_run_one_task)(
            i_e, perm_iter, fold, emotion, X_tr, Y_tr, X_te, Y_te,
            MODELS, PARAM_GRIDS, RANDOM_SEED, features_names
        )
        for (perm_iter, fold, X_tr, Y_tr, X_te, Y_te) in tasks
    )

    logging.info(f"  Finished in {time.time() - t0:.1f}s")

    # ── RESHAPE FLAT RESULTS BACK INTO emo_perm_perf[perm_iter] = [fold_perf, ...] ──
    emo_perm_perf   = [[] for _ in range(N_PERMUTATIONS)]
    emo_perm_models = [[] for _ in range(N_PERMUTATIONS)]

    for res in results:
        if res is None:
            continue  # fold was skipped (test set had <2 classes)
        perm_iter, perf, trained = res
        emo_perm_perf[perm_iter].append(perf)
        emo_perm_models[perm_iter].append(trained)

    all_emo_perm_perf.append(emo_perm_perf)
    all_models_perm.append(emo_perm_models)

logging.info("Permutation training complete. Null distributions generated.")

In [ ]:

def _save_pickle(obj, path):
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
    print(f"Saved → {path}")

_save_pickle(all_models_perm,   SAVE_PATH + f"all_model_mt_emo_vs_rest_null_{timestamp}.pkl")
_save_pickle(all_emo_perm_perf, SAVE_PATH + f"all_model_mt_modelsInfo_emo_vs_rest_null_{timestamp}.pkl")

## Load saved ouput

In [ ]:

if PID == 'emo05':
    LOAD_TIMESTAMP = '20260714'   # Emo 5
elif PID == 'emo06':
    LOAD_TIMESTAMP = '20260714'   # Emo 6 
elif PID == 'emo07':
    LOAD_TIMESTAMP = '20260714'   # Emo 7 
elif PID == 'emo01':
    LOAD_TIMESTAMP = '20260714'   # Emo 1
else:
    # Fallback to prevent a NameError
    raise ValueError(f"Unexpected or unconfigured PID: {PID}")

def _load_pickle(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

all_models_perm   = _load_pickle(SAVE_PATH + f"all_model_mt_emo_vs_rest_null_{LOAD_TIMESTAMP}.pkl")
all_emo_perm_perf = _load_pickle(SAVE_PATH + f"all_model_mt_modelsInfo_emo_vs_rest_null_{LOAD_TIMESTAMP}.pkl")

print(f"Loaded {len(all_models_perm)} emotion model sets,  {len(all_emo_perm_perf)} perf records.")

In [ ]:
def _unpack_results_null(nested_list, select_emos, drop_arrays=True):
    """
    Flatten either shape into a long-format pd.DataFrame:
        all_emo_perf:      emotion -> fold -> perf_dict
        all_emo_perm_perf: emotion -> permutation -> fold -> perf_dict

    perf_dict looks like:
        {'EN': {...metrics...}, 'SVM': {...}, 'RF': {...},
         'permutation_iter': 99}   # sibling key, not a model

    Auto-detects nesting depth per emotion by checking whether the first
    non-empty entry is a dict (fold-level, all_emo_perf) or a list
    (permutation-level, all_emo_perm_perf).
    """
    rows = []

    def _unpack_fold_dict(fold_dict, emo, fold_idx, perm_iter_fallback):
        if not fold_dict:
            return
        perm_iter = fold_dict.get('permutation_iter', perm_iter_fallback)
        for model_name, metrics in fold_dict.items():
            if model_name == 'permutation_iter':
                continue
            if not isinstance(metrics, dict):
                continue
            row = dict(metrics)
            if drop_arrays:
                row.pop('y_prob', None)
                row.pop('y_pred', None)
            row['model_name']       = model_name
            row['iter']             = f'iter_{fold_idx + 1}'
            row['emotion']          = emo
            row['permutation_iter'] = perm_iter
            rows.append(row)

    for emo, emo_entries in zip(select_emos, nested_list):
        if not emo_entries:
            continue

        # find the first non-empty entry to detect the shape
        first_non_empty = next((e for e in emo_entries if e), None)
        if first_non_empty is None:
            continue

        if isinstance(first_non_empty, dict):
            # all_emo_perf shape: emotion -> fold -> perf_dict
            for f_idx, fold_dict in enumerate(emo_entries):
                _unpack_fold_dict(fold_dict, emo, f_idx, perm_iter_fallback=None)

        elif isinstance(first_non_empty, list):
            # all_emo_perm_perf shape: emotion -> permutation -> fold -> perf_dict
            for p_idx, fold_list in enumerate(emo_entries):
                for f_idx, fold_dict in enumerate(fold_list):
                    _unpack_fold_dict(fold_dict, emo, f_idx, perm_iter_fallback=p_idx)

        else:
            continue  # unexpected shape, skip

    if not rows:
        return pd.DataFrame()

    return pd.DataFrame(rows)

In [ ]:

report_perm_df = _unpack_results_null(all_emo_perm_perf, select_emos)
models_perm_df = _unpack_results_null(all_models_perm,   select_emos)

# # Ensure SHAP arrays are numpy arrays
# for col in ('shap_vals', 'x_test'):
#     if col in report_perm_df.columns:
#         report_perm_df[col] = report_perm_df[col].apply(
#             lambda x: np.array(x) if x is not None else None
#         )

print("report_perm_df:", report_perm_df.shape)
print("models_perm_df:", models_perm_df.shape)

In [ ]:
report_perm_df.head(5)

In [ ]:
def plot_performance_null(report_df, select_emos, pid, save_path=None):
    """2×2 box-chart grid: AUC / Accuracy / Class-0 F1 / Class-1 F1.
    Each box shows the distribution across permutations (and folds) for
    that model/emotion combination — appropriate for permutation-test
    null distributions, unlike a single mean bar."""
    metrics = [
        ('auc',              'AUC — ' + pid),
        ('accuracy',         'Accuracy'),
        ('Class 0 F1-score', 'Class 0 F1-score'),
        ('Class 1 F1-score', 'Class 1 F1-score'),
    ]

    master_palette = {emo: color for emo, color in zip(ALL_EMOTIONS, EMOTION_PALETTE)}
    if 'Amusement' in master_palette:
        master_palette['Amusement'] = '#444444'
    palette = {emo: master_palette[emo] for emo in select_emos}

    fig, axes = plt.subplots(2, 2, figsize=(20, 10))
    axes = axes.flatten()

    for ax, (metric, title) in zip(axes, metrics):
        # One row per (model, emotion, permutation) — average across folds
        # (iter) only, so the box for each model/emotion reflects spread
        # across permutations, which is what you want for a null distribution.
        perm_df = (
            report_df
            .groupby(['model_name', 'emotion', 'permutation_iter'], sort=False)[metric]
            .mean().reset_index()
        )
        sns.boxplot(
            data=perm_df, x='model_name', y=metric,
            hue='emotion', hue_order=select_emos,
            ax=ax, palette=palette,
            linewidth=0.8, fliersize=2,
        )
        ax.set_title(title, fontweight='bold')
        ax.axhline(0.5, color='black', linestyle='--', linewidth=1.5)
        ax.set_ylim(0.2, 1)
        if metric != 'accuracy':
            ax.get_legend().remove()
        else:
            ax.legend(
                bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=15,
                frameon=True, edgecolor='#cccccc',
            )

    fig.tight_layout()

    if save_path:
        timestamp = datetime.now().strftime("%Y%m%d")
        fig.savefig(save_path + f"All_model_perform_mt_eachEmoVsBase_{pid}_{timestamp}.svg", dpi=300)
    plt.show()

In [ ]:


# Assuming these are defined in your environment (included here for completeness)
EMOTION_PALETTE = [
    "#332288", "#88CCEE", "#44AA99", "#117733", "#999933", 
    "#DDCC77", "#CC6677", "#882255", "#AA4499", "#DDDDDD", 
    "#E69F00", "#56B4E9", "#D55E00", "#0072B2", "#F0E442"
]

ALL_EMOTIONS = [
    "Calmness", "Entrancement", "Confusion", "Boredom", 
    "Interest", "Satisfaction", "Relief", "Excitement", 
    "Awkwardness", "Amusement", "Adoration", "Joy", 
    "Admiration", "Awe", "AestheticAppreciation"
]

def plot_accuracy_null(report_df, select_emos, pid, save_path=None):
    """Single box-chart: Accuracy.
    The box shows the distribution across permutations (and folds) for
    that model/emotion combination — appropriate for permutation-test
    null distributions. Top and right spines are removed ('box off')."""
    
    # 1. Create a MASTER mapping of all 15 emotions to their fixed colors
    master_palette = {emo: color for emo, color in zip(ALL_EMOTIONS, EMOTION_PALETTE)}
    
    # 2. Apply visibility fix globally
    master_palette['Amusement'] = '#444444'
    
    # 3. By passing master_palette directly, Seaborn maps by dictionary key. 
    # This strictly locks colors so they NEVER shift if an emotion is missing.

    # Change to a single plot instead of a 2x2 grid
    fig, ax = plt.subplots(figsize=(12, 8))
    
    metric = 'accuracy'
    title = f'Null Distribution: Accuracy — {pid}'

    # One row per (model, emotion, permutation) — average across folds
    # (iter) only, so the box for each model/emotion reflects spread
    # across permutations, which is what you want for a null distribution.
    perm_df = (
        report_df
        .groupby(['model_name', 'emotion', 'permutation_iter'], sort=False)[metric]
        .mean().reset_index()
    )
    
    sns.boxplot(
        data=perm_df, x='model_name', y=metric,
        hue='emotion', hue_order=select_emos,
        ax=ax, palette=master_palette,
        linewidth=0.8, fliersize=2,
    )
    
    ax.set_title(title, fontweight='bold', fontsize=16, pad=15)
    ax.axhline(0.5, color='black', linestyle='--', linewidth=1.5)
    ax.set_ylim(0.35, 0.65)
    
    # Improve label styling
    ax.set_ylabel('Accuracy', fontweight='bold', fontsize=14)
    ax.set_xlabel('Model', fontweight='bold', fontsize=14)
    ax.tick_params(axis='x', labelsize=13)
    ax.tick_params(axis='y', labelsize=12)

    # Place legend outside the plot area
    ax.legend(
        bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=14,
        frameon=True, edgecolor='#cccccc', title="Emotion", title_fontsize=15
    )

    # "Box off" - Removes the top and right spines from the axes
    sns.despine(ax=ax)

    fig.tight_layout()

    if save_path:
        timestamp = datetime.now().strftime("%Y%m%d")
        fig.savefig(save_path + f"Null_Distribution_Accuracy_{pid}_{timestamp}.svg", dpi=300)
        
    plt.show()

In [ ]:
# plot_accuracy_null(report_perm_df, select_emos, PID, save_path=FIG_PATH)
plot_accuracy_null(report_perm_df, select_emos, PID)

In [ ]:
import os
import pickle
import pandas as pd

# Define your default store path and metric columns 
EN_PERF_STORE = os.path.join(DATA_ROOT, "en_permutation_metrics.pkl")
METRIC_COLS = ['auc', 'accuracy', 'Class 0 F1-score', 'Class 1 F1-score']

# ssaves the distribution for confidence interval computation
def save_en_permutation_metrics_V2(report_perm_df, pid, path=EN_PERF_STORE):
    """
    Saves BOTH the full permutation distribution (for boxplots) AND 
    a summary dataframe containing the mean and SEM of the null models.
    """
    if not isinstance(pid, str) or not pid:
        raise ValueError(f"pid must be a non-empty string, got {pid!r}")

    # Dynamically find the model column (safeguard from previous steps)
    model_col = next((c for c in report_perm_df.columns if 'model' in str(c).lower()), 'model_name')

    en = report_perm_df[report_perm_df[model_col] == 'EN'].copy()
    if en.empty:
        raise ValueError(f"No EN rows in report_perm_df column '{model_col}' — refusing to save.")

    # 1. FULL DISTRIBUTION: Group by emotion and permutation_iter, averaging across folds.
    per_iter = (en.groupby(['emotion', 'permutation_iter'], sort=False)[METRIC_COLS]
                  .mean().reset_index())
    per_iter['Model'] = 'EN'

    # 2. NULL SUMMARY: Calculate the mean and SEM across the permutation iterations
    null_summary = per_iter.groupby('emotion', sort=False)[METRIC_COLS].agg(['mean', 'sem'])
    
    # Flatten the multi-level columns (e.g., 'auc_mean', 'auc_sem')
    null_summary.columns = [f"{m}_{s}" for m, s in null_summary.columns]
    null_summary = null_summary.reset_index()
    null_summary['Model'] = 'EN'

    # --- SAVE TO STORE ---
    store = {}
    if os.path.exists(path):
        with open(path, "rb") as f:
            store = pickle.load(f)

    # Save both as a dictionary payload
    store[pid] = {
        'distribution': per_iter,
        'summary': null_summary
    }                      
    
    with open(path, "wb") as f:
        pickle.dump(store, f)

    print(f"Saved EN permutation metrics & summary for {pid!r} → {path}")
    return store[pid]


def load_en_permutation_metrics(pid=None, path=EN_PERF_STORE):
    """
    Return the whole store dict, or one PID's payload {'distribution': df, 'summary': df}.
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"No metrics store at {path}")
    with open(path, "rb") as f:
        store = pickle.load(f)
    
    if pid is None:
        return store
    if pid not in store:
        raise KeyError(f"PID {pid!r} not in store. Available: {list(store)}")
    return store[pid]


def load_en_permutation_metrics_all(path=EN_PERF_STORE):
    """
    Flatten the store into TWO tidy DataFrames: one for the raw distributions, 
    and one for the aggregated null summaries (means and sems).
    """
    store = load_en_permutation_metrics(path=path)
    dist_frames = []
    summ_frames = []
    
    for pid, data in store.items():
        # Backward compatibility check: in case previous entries were just DataFrames
        if isinstance(data, pd.DataFrame):
            d = data.copy()
            d.insert(0, 'PID', pid)
            dist_frames.append(d)
            continue
            
        # Handle the new dictionary structure
        d_dist = data['distribution'].copy()
        d_dist.insert(0, 'PID', pid)
        dist_frames.append(d_dist)
        
        d_summ = data['summary'].copy()
        d_summ.insert(0, 'PID', pid)
        summ_frames.append(d_summ)
        
    dist_all = pd.concat(dist_frames, ignore_index=True) if dist_frames else pd.DataFrame()
    summ_all = pd.concat(summ_frames, ignore_index=True) if summ_frames else pd.DataFrame()
    
    return dist_all, summ_all

In [ ]:
save_en_permutation_metrics_V2(report_perm_df, PID,path=EN_PERF_STORE)



In [ ]:

accuracy_dist_all_perm, accuracy_sum_all_perm = load_en_permutation_metrics_all()
accuracy_sum_all_perm.head()


## Permutation Test for Rose plots 

In [ ]:
# =============================================================================
# Wedge-level permutation statistics for the Nightingale (Borda) rose plots
# =============================================================================
#
# Test statistic = EXACTLY the plotted value. For every wedge (feature-cluster
# c within emotion-group g) the figure shows the normalized Borda consensus
# score computed by plot_network_clusters_radial_cl_V2 from `pivot_num`.
# Here the 100 label-shuffled surrogate models (models_perm_df) are pushed
# through the IDENTICAL pipeline the observed figure used:
#
#   EN weights per fold
#     -> stability filter (non-zero in >= STABILITY_THRESH of runs)      [cell "19. Stable feature extraction", threshold=0.9]
#     -> plateau trim (cumulative |importance| <= PLATEAU_FRAC)          [remove_tail_by_plateau, 0.90]
#     -> ring-1 cutoff (|importance| >= RING_FRAC * per-emotion max)     [extract_top_features_all_emotions, ring_index=0]
#     -> collapse to feature clusters (signed value with max |.|)        [cell "28"]
#     -> Borda ranks -> per-group normalized consensus score (0-100)     [plot_network_clusters_radial_cl_V2]
#
# giving a null distribution S_null(c, g) for every wedge. Observed wedges are
# tested with the exact one-sided empirical p-value
#     p = (1 + #{S_null >= S_obs}) / (N_perm + 1)     [Phipson & Smyth 2010]
# followed by Benjamini-Hochberg FDR across all drawn wedges (primary) and a
# Westfall-Young maxT FWER p-value within each group (secondary, assumption-free).
#
# Fixed-vs-refit choices (state in Methods):
#   * The feature -> cluster assignment is held FIXED at the observed mapping.
#     Label shuffling does not change feature covariance, so the clustering is
#     ancillary to the permutation and conditioning on it is valid. Features
#     selected only under the null (absent from the observed map) are routed to
#     pseudo-clusters (prefixed "~"): they compete in the Borda ranking, so
#     null scores are not inflated, but they are never reported as wedges.
#   * The emotion grouping (cluster_dict) is held fixed: the test asks "given
#     this grouping, is this wedge's consensus score larger than chance?".
#   * Each dataset (observed or one permutation) is ranked over its OWN drawn
#     clusters, exactly as the plotting function does (M = len(pivot)), so
#     score_obs below matches the plotted bar heights to machine precision.
#
# Prerequisites in the notebook namespace (run those cells first):
#   models_perm_df   - unpacked null models  (section "Load saved ouput")
#   pivot_num        - observed cluster x emotion matrix (cell "28")
#   result           - extract_top_features_all_emotions output (cell "27")
#   features_names, extract_stable_features, remove_tail_by_plateau
#   DATA_ROOT, PID   - for the cross-participant significance store at the end
#   gm_circular_emotion_cluster_ring_black_colors importable on the path
#     (this cell defines its own _resolve_category/_feature_color/_wrap_label
#     from that module -- same palette as plot_network_clusters_radial_cl_V2,
#     independent of whether that cell has already run).
#
# With N_perm = 100 the smallest attainable p is 1/101 = 0.0099 -> report
# "p < 0.01" as the floor; extend to 500-1000 permutations for finer FDR.
# =============================================================================



# BH-FDR: scipy >= 1.11 has false_discovery_control; otherwise use an inline BH.
try:
    from scipy.stats import false_discovery_control as _scipy_bh

    def bh_adjust(p):
        return _scipy_bh(np.asarray(p, dtype=float))
except ImportError:  # scipy < 1.11

    def bh_adjust(p):
        p = np.asarray(p, dtype=float)
        n = p.size
        order = np.argsort(p)
        ranked = p[order] * n / np.arange(1, n + 1)
        q = np.minimum.accumulate(ranked[::-1])[::-1]
        out = np.empty(n)
        out[order] = np.minimum(q, 1.0)
        return out


ROMAN_SUFFIX = re.compile(r"\s+(?:I{1,3}|IV|V|VI{0,3}|IX|X)$")

RING_FRAC = 0.25          # ring_index=0 in extract_top_features_all_emotions
STABILITY_THRESH = 0.9    # threshold used for en_stable_df
PLATEAU_FRAC = 0.90       # remove_tail_by_plateau total_importance_threshold
ALPHA = 0.05

# The grouping ACTUALLY used to draw the figure (copy of the dict passed to
# plot_network_clusters_radial_cl_V2 in the "EXECUTE & SAVE" cell). If you
# change it there, change it here too.
if PID == 'emo05':
    ROSE_CLUSTER_DICT = {
        'Entranced & Aesthetic': [ 'Entrancement', 'AestheticAppreciation'],
        'Amus-Int-Admi_Awe_Re_Sa/Group2' : ['Amusement','Interest', 'Admiration', 'Awe',  'Relief','Satisfaction'],
        'Disengaged': ['Confusion', 'Boredom'],
        'Joy': ['Joy'],
        'Adoration' : ['Adoration'], 
        'Excitement' : ['Excitement'],
        'Awekwardness': ['Awkwardness'],
        'calmness': ['Calmness']}




# -------------------------Emo 6-------------------------- based on rank correlaiton on trimmed data 
elif PID == 'emo06':
    ROSE_CLUSTER_DICT =  {
        ' Interest and Aesthetic/ Group 1': ['Amusement', 'Entrancement', 'AestheticAppreciation', 'Interest', 'Satisfaction'],
        'Disengaged (Confusion-Awkwardness-Boredom)/Group 2': ['Boredom','Confusion', 'Awkwardness'],
        'Relief-Calmness/Group 3': ['Relief','Calmness'],
        'Excitement/Group 4': ['Excitement'],
        'Joy-Adoration-Admiration/Group 5' : ['Joy','Adoration','Admiration'], 
        'Awe/Group 6': ['Awe']
        }

    
# -------------------------Emo 1-------------------------- based on rank correlaiton on trimmed data 
elif PID == 'emo01':
    ROSE_CLUSTER_DICT = {
        'Excitement-Relief/Group 1': ['Excitement','Relief'],
        'Calmness-Entrancement-Aesthetic/Group 2' : ['Entrancement','Calmness','AestheticAppreciation'],
        'Interest and Awe/ Group 3': ['Awe', 'Satisfaction', 'Interest'],
        'engaged/Group 4': ['Amusement',  'Adoration'],
        'Admiration/Group 5': ['Admiration'],
        'Boredom /Group 6': ['Boredom'],
        'Awekwardness-Joy-Confusion/Group 7': ['Awkwardness', 'Joy','Confusion']
        }      




# ── 1. Fixed feature -> cluster_base map from the OBSERVED analysis ──────────

def build_feature_cluster_map(result_long_df):
    """feature -> cluster_base lookup, held constant across permutations.

    A feature can appear under different Roman-numeral sub-clusters across
    emotions; after stripping the suffix these collapse. If a feature still
    maps to more than one base, the modal base is used.
    """
    tmp = result_long_df[["feature", "cluster_name"]].copy()
    tmp["cluster_base"] = tmp["cluster_name"].str.replace(
        ROMAN_SUFFIX, "", regex=True)
    modal = (tmp.groupby("feature")["cluster_base"]
                .agg(lambda s: s.value_counts().index[0]))
    return modal.to_dict()


def _fallback_cluster(feature_name):
    """Pseudo-cluster for features selected only under the null.

    Uses the rose plot's own category resolver when available so related
    null-only features aggregate sensibly; the "~" prefix marks them as
    non-reported competitors in the Borda ranking.
    """
    try:
        return "~" + _resolve_category(feature_name)
    except NameError:
        return "~Other"


FEATURE_CLUSTER_MAP = build_feature_cluster_map(result["long_df"])
print(f"Fixed map: {len(FEATURE_CLUSTER_MAP)} observed features -> "
      f"{len(set(FEATURE_CLUSTER_MAP.values()))} cluster bases")


# ── 2. One function = the whole rose pipeline (stable_df -> wedge scores) ────

def stable_df_to_cluster_pivot(stable_df, feature_cluster_map,
                               ring_frac=RING_FRAC):
    """Replicate long_df -> agg -> pivot_num of the observed analysis.

    Applies the per-emotion ring-1 cutoff (|imp| >= ring_frac * max|imp|),
    maps features to their fixed cluster_base, and keeps the signed importance
    with maximum absolute magnitude per (emotion, cluster). Rows = clusters
    drawn in THIS dataset (same convention as the figure's pivot_num).
    """
    feat_cols = [c for c in stable_df.columns if c.endswith("_Stable_Feature")]
    records = []
    for fc in feat_cols:
        emo = fc.replace("_Stable_Feature", "")
        ic = f"{emo}_Mean_Importance"
        sub = stable_df[[fc, ic]].dropna()
        if sub.empty:
            continue
        thr = ring_frac * sub[ic].abs().max()
        sub = sub[sub[ic].abs() >= thr]
        for feat, imp in zip(sub[fc], sub[ic]):
            records.append({
                "emotion": emo,
                "cluster_base": feature_cluster_map.get(
                    feat, _fallback_cluster(feat)),
                "importance": imp,
            })
    if not records:
        return pd.DataFrame()

    long_df = pd.DataFrame(records)
    agg = (long_df.groupby(["emotion", "cluster_base"], as_index=False)
                  .agg(max_importance=("importance",
                                       lambda x: x.loc[x.abs().idxmax()])))
    return agg.pivot_table(index="cluster_base", columns="emotion",
                           values="max_importance", aggfunc="first")


def borda_group_scores(pivot, cluster_dict):
    """EXACTLY the statistic drawn by plot_network_clusters_radial_cl_V2.

    Rank clusters by |value| within each emotion (rank 1 = largest), award
    points = M - rank + 1 with M = len(pivot) (clusters drawn in this
    dataset); absent (NaN) clusters get 0. Per group: sum points over member
    emotions and normalize by M * n_members * 100.
    """
    if pivot is None or len(pivot) == 0:
        return pd.DataFrame(columns=list(cluster_dict))
    ranks = pivot.abs().rank(ascending=False)
    max_features = len(pivot)
    points = (max_features - ranks + 1).fillna(0)

    out = {}
    for gname, members in cluster_dict.items():
        valid = [e for e in members if e in points.columns]
        if not valid:
            out[gname] = pd.Series(0.0, index=pivot.index)
            continue
        max_possible = max_features * len(valid)
        out[gname] = points[valid].sum(axis=1) / max_possible * 100.0
    return pd.DataFrame(out)


def perm_df_to_trimmed_stable(one_perm_df):
    """model rows of ONE permutation -> stability-filtered, plateau-trimmed df."""
    stable_df = extract_stable_features(
        one_perm_df, features_names, model_name="EN",
        threshold=STABILITY_THRESH)
    if stable_df.empty:
        return stable_df
    return remove_tail_by_plateau(
        stable_df, total_importance_threshold=PLATEAU_FRAC)


# ── 3. Observed scores + null distributions + exact empirical p-values ───────

def rose_permutation_test(pivot_num_obs, models_perm_df, cluster_dict,
                          alpha=ALPHA, tail="greater", verbose_every=10):
    """Wedge-level permutation test.

    tail : {"greater", "two-sided"}
        "greater" (default) tests whether the wedge's consensus score is
        larger than expected by chance -- the natural direction for this
        statistic, since a Borda score only means "how prominently this
        cluster is drawn" and there is no meaningful sense in which it is
        being "pushed down" by the pipeline.
        "two-sided" additionally flags wedges that are unusually SMALL
        relative to the null (drawn less prominently than chance would
        predict), using the standard 2*min(p_upper, p_lower) construction
        (capped at 1) for both the per-wedge p-value and the Westfall-Young
        maxT correction (which then uses the null's per-permutation extreme
        deviation from its own mean, in either direction).
    """
    if tail not in ("greater", "two-sided"):
        raise ValueError('tail must be "greater" or "two-sided"')

    obs_scores = borda_group_scores(pivot_num_obs, cluster_dict)
    clusters = list(obs_scores.index)
    groups = list(cluster_dict)
    obs = obs_scores[groups].to_numpy()

    perm_ids = np.sort(models_perm_df["permutation_iter"].unique())
    n_perm = len(perm_ids)
    null_scores = np.zeros((n_perm, len(clusters), len(groups)))
    perm_trimmed_cache = {}   # reused by the per-emotion test in the next cell

    for k, p_id in enumerate(perm_ids):
        one = models_perm_df[models_perm_df["permutation_iter"] == p_id]
        trimmed = perm_df_to_trimmed_stable(one)
        perm_trimmed_cache[p_id] = trimmed
        if trimmed.empty:
            continue   # this permutation drew nothing -> all wedge scores 0
        s = borda_group_scores(
            stable_df_to_cluster_pivot(trimmed, FEATURE_CLUSTER_MAP),
            cluster_dict)
        # Clusters absent in this permutation score 0 (same "not drawn"
        # convention as the figure).
        null_scores[k] = (s.reindex(index=clusters, columns=groups)
                           .fillna(0.0).to_numpy())
        if verbose_every and (k + 1) % verbose_every == 0:
            print(f"  permutation {k + 1}/{n_perm}")

    # Exact empirical p with the +1 correction (never returns 0).
    p_upper = (1 + (null_scores >= obs[None, :, :]).sum(axis=0)) / (n_perm + 1)
    if tail == "greater":
        p_perm = p_upper
    else:  # two-sided: 2 * min(upper-tail p, lower-tail p), capped at 1
        p_lower = (1 + (null_scores <= obs[None, :, :]).sum(axis=0)) \
            / (n_perm + 1)
        p_perm = np.minimum(1.0, 2 * np.minimum(p_upper, p_lower))

    # Westfall-Young maxT: within each group, compare each observed wedge to
    # the null distribution of the most extreme wedge score across clusters.
    if tail == "greater":
        null_extreme = null_scores.max(axis=1)                # (n_perm, n_groups)
        p_fwer = (1 + (null_extreme[:, None, :] >= obs[None, :, :]).sum(axis=0)) \
            / (n_perm + 1)
    else:
        # Two-sided maxT: per permutation, take the cluster with the largest
        # ABSOLUTE deviation from that null's own mean (in either direction),
        # then compare |obs - null_mean| against that extreme.
        null_mean_cg = null_scores.mean(axis=0, keepdims=True)         # (1, C, G)
        null_dev = np.abs(null_scores - null_mean_cg)                  # (P, C, G)
        null_extreme = null_dev.max(axis=1)                            # (P, G)
        obs_dev = np.abs(obs - null_mean_cg[0])                        # (C, G)
        p_fwer = (1 + (null_extreme[:, None, :] >= obs_dev[None, :, :]).sum(axis=0)) \
            / (n_perm + 1)

    rows = []
    for i, cl in enumerate(clusters):
        for j, g in enumerate(groups):
            rows.append({
                "cluster": cl,
                "emotion_group": g,
                "score_obs": obs[i, j],
                "null_mean": null_scores[:, i, j].mean(),
                "null_p95": np.percentile(null_scores[:, i, j], 95),
                "null_p5": np.percentile(null_scores[:, i, j], 5),
                "p_perm": p_perm[i, j],
                "p_fwer_maxT": p_fwer[i, j],
            })
    stats_df = pd.DataFrame(rows)
    stats_df.attrs["tail"] = tail

    # BH-FDR across every DRAWN wedge within the participant (score_obs > 0);
    # undrawn wedges are not hypotheses.
    drawn = stats_df["score_obs"] > 0
    stats_df["q_fdr"] = np.nan
    stats_df.loc[drawn, "q_fdr"] = bh_adjust(stats_df.loc[drawn, "p_perm"])
    stats_df["sig_fdr"] = stats_df["q_fdr"] < alpha
    stats_df["sig_fwer"] = drawn & (stats_df["p_fwer_maxT"] < alpha)

    return stats_df, obs_scores, null_scores, perm_trimmed_cache


# tail="greater" (default) tests "is this wedge bigger than chance?" -- the
# natural question for a rose plot. Switch to tail="two-sided" if you also
# want to flag wedges that are unusually SMALL relative to the null.
ROSE_TEST_TAIL = "greater"

rose_stats_df, rose_obs_scores, rose_null_scores, PERM_TRIMMED_CACHE = \
    rose_permutation_test(pivot_num, models_perm_df, ROSE_CLUSTER_DICT,
                          tail=ROSE_TEST_TAIL)

rose_sig_table = (rose_stats_df[rose_stats_df["score_obs"] > 0]
                  .sort_values(["emotion_group", "p_perm"])
                  .reset_index(drop=True))
display(rose_sig_table)
rose_sig_table.round(4).to_csv(
    SAVE_PATH + f"rose_wedge_permutation_stats_{PID}_{timestamp}.csv",
    index=False)
print(f"n permutations = {rose_null_scores.shape[0]}; "
      f"p-value floor = {1 / (rose_null_scores.shape[0] + 1):.4f}")


# ── 4. Annotated rose: significance dots + null 95th-percentile arcs ─────────
#
# Color/label helpers copied verbatim from the rose-plot cell (the one that
# defines plot_network_clusters_radial_cl_V2) so this cell is self-contained
# and always resolves the SAME color for the SAME cluster name, regardless
# of what has or hasn't run earlier in the kernel. Colors are keyed purely
# off the cluster-name string via categorize_feature() -> CATEGORY_COLORS,
# so "Blood Pressure" (etc.) maps to the identical color for every PID --
# nothing here depends on rank order or any per-participant data.
from circular_emotion_cluster_ring_black_colors import (
    CATEGORY_COLORS,
    CATEGORY_ORDER,
    categorize_feature,
    _dominant_region,
    _shade_for_region,
)

_HR_TOKENS = {"hr", "hrv", "ecg", "ibi", "bpm", "rri", "nni"}
_HR_SUBSTR = ("heart", "cardio", "vascular", "pulse",
              "rmssd", "sdnn", "pnn", "tachogram")
_BP_HINTS = ("_bp", "bp_", "sbp", "dbp", "systol", "diastol",
             "blood_pressure", "pressure")

_COLOR_OVERRIDES = {
    "Heart Rate": "#0A3B2A",   # very dark green
}

_FRONTAL_HINTS = ("_front", "front", "anterior", "prefront")
_POSTERIOR_HINTS = ("_post", "post", "occip", "pariet")
_MID_HINTS = ("_mid", "mid", "central", "vertex")


def _resolve_category(feature_name):
    cat = categorize_feature(feature_name)
    n = feature_name.lower()
    if cat == "EEG Alpha":
        tokens = set(re.split(r"[^a-z0-9]+", n))
        if ("low" in tokens or "lowalpha" in n or "alphalow" in n
                or "alpha_low" in n or "low_alpha" in n):
            return "EEG Alpha-Low"
        return cat
    if cat != "Other":
        return cat
    tokens = set(re.split(r"[^a-z0-9]+", n))
    if tokens & _HR_TOKENS or any(h in n for h in _HR_SUBSTR):
        return "Heart Rate"
    if any(h in n for h in _BP_HINTS):
        return "BP"
    return cat


def _resolve_region(feature_name):
    region = _dominant_region([feature_name])
    if region is not None:
        return region
    n = feature_name.lower()
    if any(h in n for h in _FRONTAL_HINTS):
        return "Frontal"
    if any(h in n for h in _POSTERIOR_HINTS):
        return "Posterior"
    if any(h in n for h in _MID_HINTS):
        return "Mid"
    return None


def _feature_color(feature_name):
    cat = _resolve_category(feature_name)
    base = _COLOR_OVERRIDES.get(cat, CATEGORY_COLORS.get(cat, "#888888"))
    if cat.startswith("EEG "):
        region = _resolve_region(feature_name)
        base = _shade_for_region(base, region)
    return base


def _wrap_label(feature):
    return "\n".join(str(feature).split())


def plot_network_clusters_radial_sig(df_weights, cluster_dict, stats_df,
                                           alpha=ALPHA, save_path=None,
                                           show_category_legend=True,
                                           show_null_arcs=True):
    """plot_network_clusters_radial + permutation-test annotations.

    Per wedge: a dashed gray arc at the null 95th percentile (chance level the
    bar must clear) and a filled black dot at the tip of BH-FDR-significant
    wedges (open dot = additionally survives Westfall-Young maxT FWER).
    Everything else is identical to the original function.
    """
    df_abs = df_weights.abs()
    ranks = df_abs.rank(ascending=False)
    max_features = len(df_weights)
    points_df = (max_features - ranks + 1).fillna(0)

    global_scores = points_df.sum(axis=1).sort_values(ascending=False)
    global_features = global_scores[global_scores > 0].index.tolist()
    num_vars = len(global_features)
    # Identical color computation to plot_network_clusters_radial_cl_V2.
    feature_colors = [_feature_color(f) for f in global_features]

    angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
    width = 2 * np.pi / num_vars

    cols = min(3, len(cluster_dict))
    rows = int(np.ceil(len(cluster_dict) / cols))
    fig, axes = plt.subplots(rows, cols, subplot_kw={'projection': 'polar'},
                             figsize=(10 * cols, 10 * rows))
    axes = np.array(axes).flatten()

    LABEL_R, AXIS_RMAX, TITLE_Y = 108, 130, 1.10

    for ax, (cluster_name, emotions) in zip(axes, cluster_dict.items()):
        valid = [e for e in emotions if e in points_df.columns]
        if not valid:
            ax.set_title(f"{cluster_name}\n(No Data)", y=TITLE_Y,
                         fontweight='bold', fontsize=20)
            ax.set_yticklabels([])
            ax.set_xticklabels([])
            continue

        cluster_score = points_df[valid].sum(axis=1)
        normalized = cluster_score / (max_features * len(valid)) * 100
        scores_array = np.array(
            [normalized.get(f, 0) for f in global_features])

        ax.bar(angles, scores_array, width=width, bottom=0.0, alpha=0.85,
               edgecolor='white', color=feature_colors)
        ax.set_theta_offset(np.pi / 2)
        ax.set_theta_direction(-1)

        # ── Permutation-test overlays ────────────────────────────────────
        gstats = (stats_df[stats_df["emotion_group"] == cluster_name]
                  .set_index("cluster"))
        for angle, feat, score in zip(angles, global_features, scores_array):
            if feat not in gstats.index or score <= 0:
                continue
            row = gstats.loc[feat]
            if show_null_arcs and row["null_p95"] > 0:
                arc = np.linspace(angle - width / 2, angle + width / 2, 20)
                ax.plot(arc, np.full_like(arc, row["null_p95"]),
                        color="#555555", lw=1.1, ls="--", zorder=4)
            if bool(row.get("sig_fdr", False)):
                # Large marker at the wedge tip: white halo first for
                # contrast against any bar color, then the black-edged
                # marker on top.
                marker_fill = ('black' if bool(row.get("sig_fwer", False))
                               else 'white')
                r_dot = min(score + 6, 104)
                

                ax.plot(angle, r_dot, marker="*", ms=18,
                                    markerfacecolor="white", markeredgecolor="none",
                                    linestyle="none", zorder=5)
                                
                ax.plot(angle, r_dot, marker="*", ms=17,
                        markerfacecolor=marker_fill, markeredgecolor="black",
                        markeredgewidth=1.4, linestyle="none", zorder=6)

        ax.set_ylim(0, AXIS_RMAX)
        ax.set_yticks([25, 50, 75, 100])
        ax.set_yticklabels(['25', '50', '75', '100'], color='grey', size=9)
        ax.spines['polar'].set_visible(False)
        ax.plot(np.linspace(0, 2 * np.pi, 200), [100] * 200,
                color='#bbbbbb', linewidth=0.9, zorder=1)
        ax.set_title(f"{cluster_name}", y=TITLE_Y,
                     fontweight='bold', fontsize=20)
        ax.set_xticks(angles)
        ax.set_xticklabels([])

        for angle, feature in zip(angles, global_features):
            theta_deg = np.degrees(np.pi / 2 - angle) % 360
            if 90 < theta_deg < 270:
                ha, rot = 'right', theta_deg + 180
            else:
                ha, rot = 'left', theta_deg
            ax.text(angle, LABEL_R, _wrap_label(feature), ha=ha, va='center',
                    rotation=rot, rotation_mode='anchor', fontsize=18)

    for i in range(len(cluster_dict), len(axes)):
        fig.delaxes(axes[i])

    handles = [
        plt.Line2D([], [], ls="--", color="#555555",
                   label="null 95th percentile (chance level)"),
        plt.Line2D([], [], marker="o", ls="none", ms=15,
                   markerfacecolor="white", markeredgecolor="black",
                   markeredgewidth=2.0,
                   label=f"BH-FDR q < {alpha:g} (label permutation)"),
        plt.Line2D([], [], marker="o", ls="none", ms=15,
                   markerfacecolor="black", markeredgecolor="black",
                   markeredgewidth=2.0,
                   label=f"+ maxT FWER p < {alpha:g}"),
    ]
    if show_category_legend:
        present = {_resolve_category(f) for f in global_features}
        ordered = [c for c in CATEGORY_ORDER if c in present]
        ordered += [c for c in present if c not in ordered]
        handles += [
            mpatches.Patch(
                facecolor=_COLOR_OVERRIDES.get(
                    c, CATEGORY_COLORS.get(c, "#888888")),
                edgecolor="white", label=c)
            for c in ordered
        ]
    fig.legend(handles=handles, loc="lower center",
               ncol=min(len(handles), 5), fontsize=16,
               bbox_to_anchor=(0.5, -0.01), framealpha=0.95,
               edgecolor="#cccccc")

    plt.tight_layout(pad=2.5, h_pad=3.5, w_pad=2.5)
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight',
                    facecolor='white')
        print(f"Figure successfully saved to: {save_path}")
    plt.show()
    return fig


_rose_sig_fig = plot_network_clusters_radial_cl(
    pivot_num, ROSE_CLUSTER_DICT, rose_stats_df,
    save_path=FIG_PATH + (f"emotion_cluster_rank_rose_permstats_"
                          f"{PID}_{timestamp}.svg"))

# Caption text: "Dashed arcs mark each wedge's 95th percentile under 100
# label-permutation surrogates run through the identical stability -> trim ->
# ring -> Borda pipeline; dots mark wedges whose consensus score exceeds the
# null (one-sided empirical p, BH-FDR q < 0.05 across drawn wedges within the
# participant; filled dots additionally survive Westfall-Young maxT FWER)."


# =============================================================================
# Cross-participant store: BH-FDR-significant Borda wedges (one-sided)
# =============================================================================
# Same pattern as the other PID-keyed stores in this notebook (e.g. cell
# "feature_rank_corr_store", "EN_performance_metrics_allemo_allpid"): one
# shared pickle at DATA_ROOT, keyed by PID, upserted per participant, with a
# save/load/load_all trio so results can be aggregated across the cohort
# later without re-running any permutations.
ROSE_SIG_STORE_PATH = os.path.join(DATA_ROOT, "rose_sig_wedges_store.pkl")


def save_rose_sig_wedges(stats_df, pid, path=ROSE_SIG_STORE_PATH):
    """Upsert this participant's BH-FDR-significant Borda wedges into the
    shared {PID: DataFrame} store, for cross-participant aggregation.

    Only keeps rows with sig_fdr == True (one-sided, tail='greater' -- the
    wedge's consensus score is significantly LARGER than the label-permuted
    null). Refuses to save if stats_df was computed with tail='two-sided',
    since this store is specifically for the one-sided FDR result.
    """
    if not isinstance(pid, str) or not pid:
        raise ValueError(f"pid must be a non-empty string, got {pid!r}")

    tail_used = stats_df.attrs.get("tail", "greater")
    if tail_used != "greater":
        raise ValueError(
            f"Refusing to save: stats_df was computed with tail={tail_used!r}. "
            "This store holds one-sided ('greater') FDR results only -- "
            "re-run rose_permutation_test(..., tail='greater') first."
        )

    sig = (stats_df.loc[stats_df["sig_fdr"],
                        ["cluster", "emotion_group", "score_obs", "null_mean",
                         "null_p95", "p_perm", "p_fwer_maxT", "q_fdr",
                         "sig_fwer"]]
                   .reset_index(drop=True))

    store = {}
    if os.path.exists(path):
        with open(path, "rb") as f:
            store = pickle.load(f)

    store[pid] = sig                            # replace just this PID
    with open(path, "wb") as f:
        pickle.dump(store, f)

    print(f"Saved {len(sig)} FDR-significant rose wedges for {pid!r} → {path}")
    print(f"PIDs in store: {list(store)}")
    return sig


def load_rose_sig_wedges(pid=None, path=ROSE_SIG_STORE_PATH):
    """Return the whole {PID: DataFrame} dict, or one PID's DataFrame.
    A missing PID raises KeyError -- never returns the wrong one."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"No rose-wedge significance store at {path}")
    with open(path, "rb") as f:
        store = pickle.load(f)
    if pid is None:
        return store
    if pid not in store:
        raise KeyError(f"PID {pid!r} not in store. Available: {list(store)}")
    return store[pid]


def load_rose_sig_wedges_all(path=ROSE_SIG_STORE_PATH):
    """Flatten the store into one tidy DataFrame with a PID column -- e.g.
    to count how many participants show a significant Blood Pressure wedge
    in the Adoration group."""
    store = load_rose_sig_wedges(path=path)
    cols = ["PID", "cluster", "emotion_group", "score_obs", "null_mean",
            "null_p95", "p_perm", "p_fwer_maxT", "q_fdr", "sig_fwer"]
    frames = []
    for pid, df in store.items():
        d = df.copy()
        d.insert(0, "PID", pid)
        frames.append(d)
    if not frames:
        return pd.DataFrame(columns=cols)
    return pd.concat(frames, ignore_index=True)[cols]


rose_sig_saved = save_rose_sig_wedges(rose_stats_df, PID)
display(rose_sig_saved)

In [ ]:
ROSE_SIG_STORE_PATH = os.path.join(DATA_ROOT, "rose_sig_wedges_store.pkl")


def load_rose_sig_wedges(pid=None, path=ROSE_SIG_STORE_PATH):
    """Return the whole {PID: DataFrame} dict, or one PID's DataFrame.
    A missing PID raises KeyError -- never returns the wrong one."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"No rose-wedge significance store at {path}")
    with open(path, "rb") as f:
        store = pickle.load(f)
    if pid is None:
        return store
    if pid not in store:
        raise KeyError(f"PID {pid!r} not in store. Available: {list(store)}")
    return store[pid]


def load_rose_sig_wedges_all(path=ROSE_SIG_STORE_PATH):
    """Flatten the store into one tidy DataFrame with a PID column -- e.g.
    to count how many participants show a significant Blood Pressure wedge
    in the Adoration group."""
    store = load_rose_sig_wedges(path=path)
    cols = ["PID", "cluster", "emotion_group", "score_obs", "null_mean",
            "null_p95", "p_perm", "p_fwer_maxT", "q_fdr", "sig_fwer"]
    frames = []
    for pid, df in store.items():
        d = df.copy()
        d.insert(0, "PID", pid)
        frames.append(d)
    if not frames:
        return pd.DataFrame(columns=cols)
    return pd.concat(frames, ignore_index=True)[cols]


In [ ]:
Rose_all = load_rose_sig_wedges_all(path=ROSE_SIG_STORE_PATH)